<a href="https://colab.research.google.com/github/samer-glitch/TADP-Enhancing-Trustworthiness-in-Federated-Learning-Through-Provenance-Driven-Governance/blob/main/4_tadp_hps_28_adult_synthetic_documentary_evidence_stability_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""TADP-HPS-28 Adult — verified documentary stability and ±1 counterfactual audit.

Uses the archived Adult client decisions to keep the ORIGINAL 8 D2 scores,
client partitions, degradation conditions, HPS policy and five training seeds
fixed. It redraws ONLY the 20 documentary factor scores (2,000 draws by default).

Three complementary checks:
  reviewer_pm1: exhaustive single-factor documentary ±1 changes for the SAME archived cases.

Two documentary-resampling checks:
  within_tier: retain each client's original high/medium/low/missing tier,
               resample the 20 rubric levels inside that tier.
  new_profiles: resample both tier and levels using the ORIGINAL 52/33/10/5
                tier probabilities, as in the original make_manifest().

The original full training benchmark is NOT rerun or changed. No TEST data,
model fitting, or independent verification of documentary claims is involved.

Source: 1_adult_hps28.py (the original user's Adult benchmark).
Requirements: Python standard library plus NumPy; works with ZIP directly.

Example:
  python 04_tadp_experiment4_verified_colab.py \
    --archive TADP_HPS28_Adult_Phased_Results_v5.zip \
    --draws 2000 --output TADP_Experiment4_Verified_v2
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import random
import re
import sys
import statistics
import zipfile
from collections import defaultdict
from pathlib import Path

FACTORS = {
    "D1": ("source_reputation", "data_controller", "data_objective", "data_collection_lineage"),
    "D2": ("completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
           "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
           "structural_constraint_integrity"),
    "D3": ("data_dictionary", "version_logs", "collection_protocol", "definition_updates"),
    "D4": ("data_freshness", "scheduled_refresh", "retention_clarity"),
    "D5": ("regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"),
    "D6": ("license_terms", "ethical_reviews", "redistribution", "user_agreements"),
}
WEIGHTS = {"D1": .25, "D2": .15, "D3": .10, "D4": .10, "D5": .30, "D6": .10}
CRITICAL = ("D1", "D2", "D5", "D6")
D2_FACTORS = set(FACTORS["D2"])
DOC_FACTORS = set(f for dim, fs in FACTORS.items() if dim != "D2" for f in fs)
ALL_FACTORS = set.union(D2_FACTORS, DOC_FACTORS)
BANDS = {"high": (4, 5), "medium": (3, 4), "low": (1, 2), "missing": (0, 1)}
BAND_WEIGHTS = (.52, .33, .10, .05)
BASE_EVIDENCE_SEED = 7702
FL_SEEDS = (42, 43, 44, 45, 46)
ALPHAS = ("0.5", "0.1")
CONDITIONS = ("clean", "degraded")


def original_manifest_digest(manifest):
    body = {k: v for k, v in manifest.items() if k != "sha256"}
    raw = json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


def generate_docs(ids, seed, profile_map=None):
    """Replicates EXACTLY the score-drawing order in the user's make_manifest()."""
    rng = random.Random(seed)
    out = {}
    for cid in sorted(ids):
        tier = (profile_map[cid] if profile_map is not None
                else rng.choices(list(BANDS), weights=BAND_WEIGHTS)[0])
        lo, hi = BANDS[tier]
        scores = {factor: rng.randint(lo, hi) for factor in sorted(DOC_FACTORS)}
        out[cid] = {"tier": tier, "scores": scores}
    return out


def evaluate_hps(scores, policy):
    """Replicates the original evaluate() admission logic (no ablations)."""
    if set(scores) != ALL_FACTORS:
        raise ValueError("Expected 28 exact factor scores")
    if any(type(s) is not int or not (0 <= s <= 5) for s in scores.values()):
        raise ValueError("Invalid factor score")
    dims = {d: sum(scores[f] for f in fs) / len(fs) for d, fs in FACTORS.items()}
    hps = sum(WEIGHTS[d] * dims[d] for d in FACTORS)
    review = sum(WEIGHTS[d] * dims[d] for d in CRITICAL) / sum(WEIGHTS[d] for d in CRITICAL)
    if any(s == 0 for s in scores.values()):
        return False, "BLOCKED_ZERO_FACTOR", hps
    if any(value < policy["min_dimension"] for value in dims.values()):
        return False, "BLOCKED_DIMENSION_FLOOR", hps
    for factor, minimum in policy.get("critical_factor_floors", {}).items():
        if scores[factor] < minimum:
            return False, "BLOCKED_CRITICAL_FACTOR", hps
    if hps < policy["reject_below"]:
        return False, "AUTO_REJECT", hps
    if hps >= policy["direct_accept_from"]:
        return True, "AUTO_ACCEPT", hps
    review_threshold = (policy["reject_below"] + policy["direct_accept_from"]) / 2
    return (True, "REVIEW_ACCEPT", hps) if review >= review_threshold else (False, "REVIEW_REJECT", hps)


def read_archive(archive_path):
    """Load only existing decisions and original manifest, directly from ZIP."""
    pattern = re.compile(r"/adult_alpha(0\.5|0\.1)/decisions_adult_seed(4[2-6])_(clean|degraded)_none\.json$")
    by_condition = {}
    manifests = {}
    with zipfile.ZipFile(archive_path) as zf:
        paths = zf.namelist()
        for path in paths:
            m = pattern.search("/" + path.lstrip("/"))
            if m:
                key = (m.group(1), int(m.group(2)), m.group(3))
                if key in by_condition:
                    raise ValueError(f"Duplicate archived condition: {key}")
                by_condition[key] = json.loads(zf.read(path))
        for alpha in ALPHAS:
            hits = [p for p in paths if p.endswith(f"adult_alpha{alpha}/manifest_adult_K50_E7702.json")]
            if len(hits) != 1:
                raise ValueError(f"Expected one original Adult manifest for alpha={alpha}; found {len(hits)}")
            manifests[alpha] = json.loads(zf.read(hits[0]))

    expected = {(a, s, c) for a in ALPHAS for s in FL_SEEDS for c in CONDITIONS}
    if set(by_condition) != expected:
        raise ValueError(f"Missing conditions: {sorted(expected - set(by_condition))}; "
                         f"unexpected: {sorted(set(by_condition) - expected)}")
    if manifests["0.5"]["sha256"] != manifests["0.1"]["sha256"]:
        raise ValueError("Different documentary manifests across alphas; investigate before pairing")
    manifest = manifests["0.5"]
    if original_manifest_digest(manifest) != manifest["sha256"]:
        raise ValueError("Original documentary manifest hash check failed")
    return by_condition, manifest


def validate_and_prepare(archived, original_manifest):
    """Fail rather than silently changing the original experiment."""
    ids = sorted(original_manifest["clients"])
    if len(ids) != 50:
        raise ValueError(f"Expected 50 clients, found {len(ids)}")
    fixed_tiers = {cid: original_manifest["clients"][cid]["tier"] for cid in ids}
    original_docs = generate_docs(ids, BASE_EVIDENCE_SEED)
    checks = 0
    all_d2 = {}
    for key, run in sorted(archived.items()):
        if sorted(run["decisions"]) != ids:
            raise ValueError(f"Different clients for {key}")
        if run["manifest_sha256"] != original_manifest["sha256"]:
            raise ValueError(f"Original manifest hash differs for {key}")
        policy = run["policy"]
        if any(policy[k] != v for k, v in {"min_dimension": 2.5,
              "reject_below": 3.0, "direct_accept_from": 3.5}.items()):
            raise ValueError(f"Unexpected policy for {key}: {policy}")
        per_client = {}
        for cid, record in run["decisions"].items():
            archived_scores = record["factor_scores"]
            orig_scores = {f: original_manifest["clients"][cid]["documentary_factors"][f]["rubric_level"]
                           for f in DOC_FACTORS}
            if original_docs[cid]["scores"] != orig_scores:
                raise AssertionError(f"The original synthetic generator did not reproduce {cid}")
            if any(archived_scores[f] != orig_scores[f] for f in DOC_FACTORS):
                raise AssertionError(f"Documentary scores do not match manifest for {key}, {cid}")
            accepted, path, hps = evaluate_hps(archived_scores, policy)
            if accepted != record["accepted"] or path != record["path"]:
                raise AssertionError(f"Original admission decision differs for {key}, {cid}: {path}")
            if abs(hps - record["hps"]) > 1e-8:
                raise AssertionError(f"Original HPS differs for {key}, {cid}")
            per_client[cid] = {f: archived_scores[f] for f in D2_FACTORS}
            checks += 1
        all_d2[key] = per_client
    print(f"Verified {checks} original Adult client assessments and original evidence seed 7702.")
    return ids, fixed_tiers, all_d2


def write_csv(path, records, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


def mean(values):
    return statistics.mean(values)


def sample_sd(values):
    return statistics.stdev(values) if len(values) >= 2 else 0.0


def run(archive_path, output_dir, n_draws=2000, seed_start=8801):
    """Fast, exact vectorized reassessment; reuses the *original* evaluation policy."""
    import numpy as np  # Available by default in Google Colab.
    if n_draws < 2:
        raise ValueError("At least two draws are needed to calculate sample SD.")
    if BASE_EVIDENCE_SEED in range(seed_start, seed_start + n_draws):
        raise ValueError("Do not include the original evidence seed 7702.")
    archived, manifest = read_archive(archive_path)
    ids, fixed_tiers, fixed_d2 = validate_and_prepare(archived, manifest)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # Each row is one ORIGINAL client assessment; 50 clients x 5 FL seeds
    # x 2 alphas x 2 conditions = 1,000 rows.
    cases = [(a, c, s, cid) for a in ALPHAS for c in CONDITIONS
             for s in FL_SEEDS for cid in ids]
    client_idx = np.asarray([ids.index(cid) for _, _, _, cid in cases], dtype=int)
    d2_matrix = np.asarray([[fixed_d2[(a, s, c)][cid][f] for f in FACTORS['D2']]
                            for a, c, s, cid in cases], dtype=float)
    d2_mean = d2_matrix.mean(axis=1)
    d2_has_zero = np.any(d2_matrix == 0, axis=1)
    old_accept = np.asarray([archived[(a, s, c)]['decisions'][cid]['accepted']
                             for a, c, s, cid in cases], dtype=bool)
    old_hps = np.asarray([archived[(a, s, c)]['decisions'][cid]['hps']
                          for a, c, s, cid in cases], dtype=float)
    groups = {(a, c): np.asarray([i for i, (a1, c1, _, _) in enumerate(cases)
                                  if a1 == a and c1 == c], dtype=int)
              for a in ALPHAS for c in CONDITIONS}
    for key in archived:
        if archived[key]['policy'].get('critical_factor_floors'):
            raise ValueError('This fast path needs a new implementation for nonempty critical-factor floors.')

    doc_names = sorted(DOC_FACTORS)
    doc_cols = {f: j for j, f in enumerate(doc_names)}
    base_count = {key: float(np.mean(old_accept[idx]) * len(ids))
                  for key, idx in groups.items()}
    result_rows = []
    # Validate the vectorization against the original scalar policy for the FIRST draw.
    verified_fast_path = False
    for mode in ('within_tier', 'new_profiles'):
        for draw_idx, evidence_seed in enumerate(range(seed_start, seed_start + n_draws)):
            docs = generate_docs(ids, evidence_seed,
                                 profile_map=fixed_tiers if mode == 'within_tier' else None)
            doc_array = np.asarray([[docs[cid]['scores'][f] for f in doc_names]
                                    for cid in ids], dtype=float)
            doc_means = {dim: np.mean(doc_array[:, [doc_cols[f] for f in fs]], axis=1)
                         for dim, fs in FACTORS.items() if dim != 'D2'}
            zero_documentary = np.any(doc_array == 0, axis=1)
            dimension_floor_fails = d2_mean < 2.5
            for dim in doc_means:
                dimension_floor_fails = dimension_floor_fails | (doc_means[dim][client_idx] < 2.5)
            zero_fails = d2_has_zero | zero_documentary[client_idx]
            hps = WEIGHTS['D2'] * d2_mean
            for dim, dim_means in doc_means.items():
                hps = hps + WEIGHTS[dim] * dim_means[client_idx]
            review = (WEIGHTS['D1'] * doc_means['D1'][client_idx] +
                      WEIGHTS['D2'] * d2_mean +
                      WEIGHTS['D5'] * doc_means['D5'][client_idx] +
                      WEIGHTS['D6'] * doc_means['D6'][client_idx]) / 0.80
            accepted = (~zero_fails & ~dimension_floor_fails &
                        ((hps >= 3.5) | ((hps >= 3.0) & (review >= 3.25))))

            if not verified_fast_path:
                for i in range(len(cases)):
                    a, c, s, cid = cases[i]
                    all_scores = {**fixed_d2[(a, s, c)][cid], **docs[cid]['scores']}
                    expected_accept, _, expected_hps = evaluate_hps(all_scores, archived[(a, s, c)]['policy'])
                    if bool(accepted[i]) != expected_accept or abs(float(hps[i]) - expected_hps) > 1e-9:
                        raise AssertionError(f'Vectorized policy does not match original for {a}, {c}, {s}, {cid}')
                print('Verified fast calculation exactly matches the original policy across 1,000 cases.')
                verified_fast_path = True

            for alpha in ALPHAS:
                for condition in CONDITIONS:
                    idx = groups[(alpha, condition)]
                    same = (accepted[idx] == old_accept[idx])
                    result_rows.append({
                        'mode': mode, 'evidence_seed': evidence_seed,
                        'alpha': alpha, 'condition': condition,
                        'baseline_admitted_mean_over_5_seeds': round(base_count[(alpha, condition)], 5),
                        'new_admitted_mean_over_5_seeds': round(float(accepted[idx].sum()) / len(FL_SEEDS), 5),
                        'decision_agreement_pct': round(100 * float(same.mean()), 5),
                        'mean_abs_hps_difference': round(float(np.mean(np.abs(hps[idx] - old_hps[idx]))), 6),
                        'accept_to_reject_occurrences': int((old_accept[idx] & ~accepted[idx]).sum()),
                        'reject_to_accept_occurrences': int((~old_accept[idx] & accepted[idx]).sum()),
                        'clients_compared': int(len(idx)),
                    })
        print(f'Finished {n_draws:,} documentary draws in mode: {mode}')

    summary_rows = []
    for mode in ('within_tier', 'new_profiles'):
        for alpha in ALPHAS:
            for condition in CONDITIONS:
                rr = [r for r in result_rows if r['mode'] == mode and
                      r['alpha'] == alpha and r['condition'] == condition]
                counts = [r['new_admitted_mean_over_5_seeds'] for r in rr]
                agreements = [r['decision_agreement_pct'] for r in rr]
                deltas = [r['mean_abs_hps_difference'] for r in rr]
                summary_rows.append({
                    'mode': mode, 'alpha': alpha, 'condition': condition,
                    'n_documentary_draws': len(rr), 'n_fl_seeds_fixed': len(FL_SEEDS),
                    'baseline_mean_admitted': rr[0]['baseline_admitted_mean_over_5_seeds'],
                    'new_mean_admitted': round(mean(counts), 4),
                    'new_sd_across_documentary_draws': round(sample_sd(counts), 4),
                    'new_min_mean_admitted': round(min(counts), 4),
                    'new_max_mean_admitted': round(max(counts), 4),
                    'decision_agreement_mean_pct': round(mean(agreements), 2),
                    'decision_agreement_sd_pct': round(sample_sd(agreements), 2),
                    'mean_abs_hps_difference': round(mean(deltas), 5),
                    'accept_to_reject_mean_per_draw': round(mean([r['accept_to_reject_occurrences'] for r in rr]), 2),
                    'reject_to_accept_mean_per_draw': round(mean([r['reject_to_accept_occurrences'] for r in rr]), 2),
                    'min_decision_agreement_pct': round(min(agreements), 2),
                    'max_decision_agreement_pct': round(max(agreements), 2),
                })
    write_csv(out / 'evidence_stability_summary.csv', summary_rows, list(summary_rows[0]))
    write_csv(out / 'evidence_stability_by_draw.csv', result_rows, list(result_rows[0]))
    (out / 'METHODS_README.txt').write_text(
        'Adult HPS-28 synthetic documentary evidence stability (no model retraining)\n'
        f'{n_draws:,} independent evidence draws per mode; evidence seeds {seed_start}–{seed_start+n_draws-1}.\n'
        'Fifty clients; original FL seeds 42–46 fixed; alphas 0.5/0.1; clean/degraded.\n'
        'within_tier: same documentary profile tier, redraw all 20 factor values.\n'
        'new_profiles: redraw profile tier from probabilities 52/33/10/5 and all 20 values.\n'
        'Same 8 measured D2 scores, same admission policy, same original archived cases.\n'
        'Each draw compares 250 historical client assessments per alpha-condition.\n'
        'Mean admitted = average admissions per FL seed; SD = across documentary draws\n'
        'of five-seed mean admissions, NOT independent training-seed SD.\n'
        'Decision agreement = agreement with original synthetic evidence assignment.\n'
        'NOT document verification accuracy, real-document generalization, or predictive accuracy.\n'
        'Increasing draws improves precision for the assumed synthetic generator only.\n',
        encoding='utf-8',
    )
    print(f'\nSynthetic evidence sensitivity ({n_draws:,} documentary draws per mode):')
    print(f'{"MODE":<14} {"alpha":<5} {"data":<9} {"original":>9} {"resampled mean±SD":>20} {"agreement":>11}')
    for row in summary_rows:
        print(f"{row['mode']:<14} {row['alpha']:<5} {row['condition']:<9} "
              f"{row['baseline_mean_admitted']:>9.1f} "
              f"{row['new_mean_admitted']:>8.2f}±{row['new_sd_across_documentary_draws']:<8.2f} "
              f"{row['decision_agreement_mean_pct']:>10.2f}%")
    print(f'\nSaved results: {out.resolve()}')
    return summary_rows

def _resolve_archive(path):
    """Find the archived Adult results ZIP; reject raw adult.csv explicitly."""
    path = Path(path)

    def accept(candidate):
        if candidate.suffix.lower() != ".zip" or not zipfile.is_zipfile(candidate):
            raise ValueError(
                "The experiment needs the ORIGINAL Adult RESULTS ZIP, not adult.csv. "
                "Upload TADP_HPS28_Adult_Phased_Results_v5.zip "
                "(containing archived decision JSON files)."
            )
        return candidate

    for candidate in (path, Path("/content") / path.name):
        if candidate.is_file():
            return accept(candidate)

    # Common case: the archive was already uploaded with a (1) filename suffix.
    # Use it only when a single unambiguous match exists.
    candidates = []
    for folder in (Path.cwd(), Path("/content")):
        if folder.is_dir():
            candidates.extend(p for p in folder.glob("TADP_HPS28_Adult_Phased_Results_v5*.zip")
                              if p.is_file() and zipfile.is_zipfile(p))
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if len(candidates) == 1:
        print(f"Found existing Adult results ZIP: {candidates[0].name}")
        return candidates[0]

    try:
        import google.colab  # noqa: F401
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            "Adult results ZIP not found. Run with "
            "--archive /path/to/TADP_HPS28_Adult_Phased_Results_v5.zip. "
            "The raw adult.csv dataset is NOT used in this analysis."
        )

    for attempt in range(2):
        print("\nUPLOAD THIS FILE (not adult.csv):")
        print("  TADP_HPS28_Adult_Phased_Results_v5.zip")
        print("It contains prior HPS decisions and documentary scores.")
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded
                      if name.lower().endswith(".zip") and zipfile.is_zipfile(name)]
        if len(candidates) == 1:
            return accept(candidates[0])
        print("Incorrect selection: please upload the original RESULTS .zip file. ")
        print("adult.csv is the raw dataset and cannot replace the archived decisions.")
    raise FileNotFoundError(
        "No Adult results ZIP was provided. Download or locate "
        "TADP_HPS28_Adult_Phased_Results_v5.zip and run again."
    )



# ================================================================
# ADDITIONAL AUDIT: independent, exhaustive single-factor +/-1
# ================================================================

def audit_baseline_and_one_factor(archive_path, output_dir):
    """Replay original cases; enumerate valid +/-1 documentary changes.

    IMPORTANT: This is a counterfactual scoring test, not an observed
    human-reviewer error study. Do not infer real error probabilities.
    """
    archived, manifest = read_archive(archive_path)
    ids, tiers, all_d2 = validate_and_prepare(archived, manifest)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    baseline_rows = []
    all_baseline = 0
    all_cases = 0
    for (alpha, seed, condition), data in sorted(archived.items()):
        accepted = sum(1 for rec in data['decisions'].values() if rec['accepted'])
        n = len(data['decisions'])
        if accepted != data['hps_accepted_clients']:
            raise AssertionError(f"Archived summary differs: {alpha}/{seed}/{condition}")
        baseline_rows.append({
            'alpha': alpha, 'training_seed': seed, 'condition': condition,
            'n_clients': n, 'admitted': accepted, 'rejected': n-accepted,
            'manifest_sha256': data['manifest_sha256'],
            'global_train_test_sha256': data['global_train_test_sha256'],
        })
        all_baseline += accepted
        all_cases += n
    if all_cases != 1000 or len(baseline_rows) != 20:
        raise AssertionError(f"Expected 1,000 original cases and 20 conditions, got {all_cases}/{len(baseline_rows)}")
    write_csv(out / 'baseline_admission_counts.csv', baseline_rows, list(baseline_rows[0]))

    scenario_path = out / 'reviewer_pm1_all_feasible_scenarios.csv'
    flipped_rows = []
    total_feasible = total_changes = to_accept = to_reject = 0
    vulnerable = set()
    by_factor = defaultdict(lambda: {'feasible': 0, 'flips': 0, 'to_accept': 0, 'to_reject': 0})
    fields = [
        'alpha', 'training_seed', 'condition', 'client_id', 'factor', 'delta',
        'old_score', 'new_score', 'baseline_accepted', 'counterfactual_accepted',
        'changed', 'baseline_path', 'counterfactual_path',
    ]
    with scenario_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for (alpha, seed, condition), run_data in sorted(archived.items()):
            policy = run_data['policy']
            for cid, record in sorted(run_data['decisions'].items()):
                source = record['factor_scores']
                base_accepted = bool(record['accepted'])
                for factor in sorted(DOC_FACTORS):
                    old = source[factor]
                    for delta in (-1, 1):
                        new = old + delta
                        if not 0 <= new <= 5:
                            continue
                        counter = dict(source)
                        counter[factor] = new
                        accepted, path, _ = evaluate_hps(counter, policy)
                        changed = accepted != base_accepted
                        r = {
                            'alpha': alpha, 'training_seed': seed, 'condition': condition,
                            'client_id': cid, 'factor': factor, 'delta': delta,
                            'old_score': old, 'new_score': new,
                            'baseline_accepted': base_accepted,
                            'counterfactual_accepted': accepted,
                            'changed': changed, 'baseline_path': record['path'],
                            'counterfactual_path': path,
                        }
                        writer.writerow(r)
                        total_feasible += 1
                        by_factor[factor]['feasible'] += 1
                        if changed:
                            flipped_rows.append(r)
                            total_changes += 1
                            by_factor[factor]['flips'] += 1
                            vulnerable.add((alpha, seed, condition, cid))
                            if accepted:
                                to_accept += 1
                                by_factor[factor]['to_accept'] += 1
                            else:
                                to_reject += 1
                                by_factor[factor]['to_reject'] += 1
    write_csv(out / 'reviewer_pm1_changed_scenarios.csv', flipped_rows, fields)
    factor_rows = []
    for f in sorted(DOC_FACTORS):
        r = by_factor[f]
        factor_rows.append({
            'factor': f, **r,
            'change_rate_pct': 100 * r['flips'] / r['feasible'] if r['feasible'] else 0,
        })
    write_csv(out / 'reviewer_pm1_by_factor.csv', factor_rows, list(factor_rows[0]))

    info = {
        'audit': 'exhaustive single-factor documentary +/-1 policy replay',
        'original_client_assessments': all_cases,
        'baseline_admitted': all_baseline,
        'baseline_rejected': all_cases-all_baseline,
        'baseline_admission_rate_pct': round(100*all_baseline/all_cases, 6),
        'feasible_single_factor_scenarios': total_feasible,
        'admission_decisions_changed': total_changes,
        'change_rate_of_feasible_scenarios_pct': round(100*total_changes/total_feasible, 6),
        'rejected_to_admitted': to_accept,
        'admitted_to_rejected': to_reject,
        'distinct_original_assessments_sensitive_to_at_least_one_change': len(vulnerable),
        'assessments_sensitive_pct': round(100*len(vulnerable)/all_cases, 6),
        'n_real_documentary_records': 0,
        'n_real_human_reviewers': 0,
        'warning': 'These are feasible counterfactual scenarios, not empirical reviewer error rates.',
    }
    (out / 'reviewer_pm1_summary.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
    print('\nINDEPENDENT SINGLE-FACTOR +/-1 AUDIT:')
    for k in ['original_client_assessments', 'baseline_admitted',
              'feasible_single_factor_scenarios', 'admission_decisions_changed',
              'change_rate_of_feasible_scenarios_pct',
              'distinct_original_assessments_sensitive_to_at_least_one_change']:
        print(f'  {k}: {info[k]}')
    return info


def make_zip(output_dir, zip_path, archive_path, n_draws, seed_start, pm1_info):
    """Package only the current run outputs, with provenance metadata."""
    out = Path(output_dir)
    expected_files = [
        'baseline_admission_counts.csv',
        'evidence_stability_summary.csv',
        'evidence_stability_by_draw.csv',
        'METHODS_README.txt',
        'reviewer_pm1_summary.json',
        'reviewer_pm1_by_factor.csv',
        'reviewer_pm1_changed_scenarios.csv',
        'reviewer_pm1_all_feasible_scenarios.csv',
    ]
    for fname in expected_files:
        if not (out / fname).is_file():
            raise FileNotFoundError(f'Missing output required for ZIP: {fname}')
    sha = hashlib.sha256()
    with Path(archive_path).open('rb') as fh:
        for block in iter(lambda: fh.read(1024*1024), b''):
            sha.update(block)
    _, manifest = read_archive(archive_path)
    metadata = {
        'experiment': 'TADP Experiment 4',
        'version': 'verified-v2',
        'input_archive_name': Path(archive_path).name,
        'input_archive_sha256': sha.hexdigest(),
        'original_documentary_manifest_sha256': manifest['sha256'],
        'original_documentary_seed': BASE_EVIDENCE_SEED,
        'resampled_seed_start': seed_start,
        'resampled_seed_end_inclusive': seed_start + n_draws - 1,
        'draws_per_mode': n_draws,
        'n_fl_seeds_held_fixed': 5,
        'n_alpha_condition_combinations': 4,
        'original_client_assessments': 1000,
        'doc_factors_varied': len(DOC_FACTORS),
        'd2_factors_held_fixed': len(D2_FACTORS),
        'profile_tier_probabilities': {'high':0.52,'medium':0.33,'low':0.10,'missing':0.05},
        'modes': ['within_tier', 'new_profiles'],
        'baseline_admitted': pm1_info['baseline_admitted'],
        'pm1_scenarios': pm1_info['feasible_single_factor_scenarios'],
        'pm1_changed': pm1_info['admission_decisions_changed'],
        'fl_retrained': False,
        'real_documents_reviewed': 0,
        'real_human_reviewers': 0,
        'summary_sd_definition': f'sample SD across {n_draws} documentary draws of each five-training-seed mean',
        'warning': 'Synthetic-scenario robustness does NOT validate real-world provenance measurement.',
    }
    (out / 'audit_provenance.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    expected_files.append('audit_provenance.json')
    readme = (
        'TADP EXPERIMENT 4: VERIFIED SYNTHETIC DOCUMENTARY SENSITIVITY\n'
        '==========================================================\n'
        f'Archive SHA256: {metadata["input_archive_sha256"]}\n'
        f'Original documentary manifest SHA256: {metadata["original_documentary_manifest_sha256"]}\n'
        f'Baseline: {pm1_info["baseline_admitted"]}/1000 original archived assessments accepted.\n'
        f'Resampling: {n_draws:,} documentary draws for each of within_tier and new_profiles.\n'
        'within_tier preserves each client documentary tier; redraws documentary rubric scores.\n'
        'new_profiles redraws each client tier (52/33/10/5%) and all documentary scores.\n'
        'Both modes hold original TRAIN-derived D2, five FL seeds, alphas, client datasets, and policy fixed.\n'
        f'Single-factor +/-1 replay: {pm1_info["feasible_single_factor_scenarios"]:,} feasible scenarios; '
        f'{pm1_info["admission_decisions_changed"]} admission flips; '
        f'{pm1_info["distinct_original_assessments_sensitive_to_at_least_one_change"]} vulnerable assessments.\n'
        'SD in profile summary: variation across documentary draws in the FIVE-SEED MEAN, NOT FL seed SD.\n'
        'All results are counterfactual, synthetic, and do NOT measure actual reviewer error rates.\n'
        'Zero real documents and zero real human reviewers were used.\n'
        'Outputs contain scenarios and original decision metadata, but no raw Adult dataset.\n'
    )
    (out / 'README_RESULTS.txt').write_text(readme, encoding='utf-8')
    expected_files.append('README_RESULTS.txt')
    # Include the exact executable audit script so a reviewer can inspect the
    # implementation alongside its outputs without a separate download.
    script_source = Path(__file__).resolve() if '__file__' in globals() else None
    if script_source is not None and script_source.is_file():
        (out / '04_tadp_experiment4_verified_colab.py').write_bytes(script_source.read_bytes())
        expected_files.append('04_tadp_experiment4_verified_colab.py')
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=8) as zf:
        for fname in expected_files:
            zf.write(out / fname, arcname=f'TADP_Experiment4_Verified_v2/{fname}')
    print('\nZIP CREATED:', zip_path.resolve())
    print('ZIP SIZE:', f'{zip_path.stat().st_size/1024/1024:.2f} MB')
    return zip_path


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--archive', type=Path,
                        default=Path('TADP_HPS28_Adult_Phased_Results_v5.zip'),
                        help='Original Adult results ZIP (with archived decisions + manifest)')
    parser.add_argument('--output', type=Path,
                        default=Path('TADP_Experiment4_Verified_v2'))
    parser.add_argument('--zip', dest='zip_path', type=Path,
                        default=Path('TADP_Experiment4_Verified_v2_RESULTS.zip'))
    parser.add_argument('--draws', type=int, default=2000,
                        help='Documentary draws per mode (standard 2000)')
    parser.add_argument('--seed-start', type=int, default=8801)
    parser.add_argument('--no-download', action='store_true',
                        help='Skip automatic Colab ZIP download')
    parser.add_argument('--no-upload-prompt', action='store_true',
                        help='Fail instead of prompting for the missing Adult ZIP')
    args, unexpected = parser.parse_known_args(argv)
    if unexpected and not (len(unexpected)==2 and unexpected[0]=='-f' and unexpected[1].endswith('.json')):
        parser.error(f'Unexpected arguments: {unexpected}')
    if args.no_upload_prompt:
        archive = args.archive
        if not archive.is_file() or not zipfile.is_zipfile(archive):
            raise FileNotFoundError(f'Original Adult ZIP missing or invalid: {archive}')
    else:
        archive = _resolve_archive(args.archive)
    if args.draws < 2:
        parser.error('--draws must be >= 2')
    # Original Experiment 4, unchanged to preserve reproducibility.
    run(archive, args.output, args.draws, args.seed_start)
    pm1_info = audit_baseline_and_one_factor(archive, args.output)
    final_zip = make_zip(args.output, args.zip_path, archive,
                         args.draws, args.seed_start, pm1_info)
    if not args.no_download:
        try:
            from google.colab import files as colab_files
        except ImportError:
            pass
        else:
            try:
                colab_files.download(str(final_zip.resolve()))
            except Exception as exc:
                print('Automatic download unavailable. Use Colab file sidebar:', final_zip.resolve(), exc)
    return final_zip


if __name__ == '__main__':
    main()



UPLOAD THIS FILE (not adult.csv):
  TADP_HPS28_Adult_Phased_Results_v5.zip
It contains prior HPS decisions and documentary scores.


Saving TADP_HPS28_Adult_Phased_Results_v5.zip to TADP_HPS28_Adult_Phased_Results_v5.zip
Verified 1000 original Adult client assessments and original evidence seed 7702.
Verified fast calculation exactly matches the original policy across 1,000 cases.
Finished 2,000 documentary draws in mode: within_tier
Finished 2,000 documentary draws in mode: new_profiles

Synthetic evidence sensitivity (2,000 documentary draws per mode):
MODE           alpha data       original    resampled mean±SD   agreement
within_tier    0.5   clean          30.2    30.20±0.02         100.00%
within_tier    0.5   degraded       29.4    29.54±0.14          99.49%
within_tier    0.1   clean          16.2    16.40±0.03          99.60%
within_tier    0.1   degraded       14.6    14.78±0.08          99.56%
new_profiles   0.5   clean          30.2    30.96±1.90          80.39%
new_profiles   0.5   degraded       29.4    30.23±1.86          80.58%
new_profiles   0.1   clean          16.2    16.32±1.11          90.00%
n

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""TADP-HPS-28 Adult — synthetic documentary-evidence stability experiment.

Uses the archived Adult client decisions to keep the ORIGINAL 8 D2 scores,
client partitions, degradation conditions, HPS policy and five training seeds
fixed. It redraws ONLY the 20 documentary factor scores (2,000 draws by default).

Two complementary checks:
  within_tier: retain each client's original high/medium/low/missing tier,
               resample the 20 rubric levels inside that tier.
  new_profiles: resample both tier and levels using the ORIGINAL 52/33/10/5
                tier probabilities, as in the original make_manifest().

The original full training benchmark is NOT rerun or changed. No TEST data,
model fitting, or independent verification of documentary claims is involved.

Source: 1_adult_hps28.py (the original user's Adult benchmark).
Requirements: Python standard library only; works with ZIP directly.

Example:
  python TADP_Adult_Synthetic_Evidence_Stability_2000_DRAWS.py \
    --archive TADP_HPS28_Adult_Phased_Results_v5.zip \
    --draws 2000 --output TADP_Adult_Evidence_Stability_2000
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import random
import re
import sys
import statistics
import zipfile
from collections import defaultdict
from pathlib import Path

FACTORS = {
    "D1": ("source_reputation", "data_controller", "data_objective", "data_collection_lineage"),
    "D2": ("completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
           "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
           "structural_constraint_integrity"),
    "D3": ("data_dictionary", "version_logs", "collection_protocol", "definition_updates"),
    "D4": ("data_freshness", "scheduled_refresh", "retention_clarity"),
    "D5": ("regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"),
    "D6": ("license_terms", "ethical_reviews", "redistribution", "user_agreements"),
}
WEIGHTS = {"D1": .25, "D2": .15, "D3": .10, "D4": .10, "D5": .30, "D6": .10}
CRITICAL = ("D1", "D2", "D5", "D6")
D2_FACTORS = set(FACTORS["D2"])
DOC_FACTORS = set(f for dim, fs in FACTORS.items() if dim != "D2" for f in fs)
ALL_FACTORS = set.union(D2_FACTORS, DOC_FACTORS)
BANDS = {"high": (4, 5), "medium": (3, 4), "low": (1, 2), "missing": (0, 1)}
BAND_WEIGHTS = (.52, .33, .10, .05)
BASE_EVIDENCE_SEED = 7702
FL_SEEDS = (42, 43, 44, 45, 46)
ALPHAS = ("0.5", "0.1")
CONDITIONS = ("clean", "degraded")


def original_manifest_digest(manifest):
    body = {k: v for k, v in manifest.items() if k != "sha256"}
    raw = json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


def generate_docs(ids, seed, profile_map=None):
    """Replicates EXACTLY the score-drawing order in the user's make_manifest()."""
    rng = random.Random(seed)
    out = {}
    for cid in sorted(ids):
        tier = (profile_map[cid] if profile_map is not None
                else rng.choices(list(BANDS), weights=BAND_WEIGHTS)[0])
        lo, hi = BANDS[tier]
        scores = {factor: rng.randint(lo, hi) for factor in sorted(DOC_FACTORS)}
        out[cid] = {"tier": tier, "scores": scores}
    return out


def evaluate_hps(scores, policy):
    """Replicates the original evaluate() admission logic (no ablations)."""
    if set(scores) != ALL_FACTORS:
        raise ValueError("Expected 28 exact factor scores")
    if any(type(s) is not int or not (0 <= s <= 5) for s in scores.values()):
        raise ValueError("Invalid factor score")
    dims = {d: sum(scores[f] for f in fs) / len(fs) for d, fs in FACTORS.items()}
    hps = sum(WEIGHTS[d] * dims[d] for d in FACTORS)
    review = sum(WEIGHTS[d] * dims[d] for d in CRITICAL) / sum(WEIGHTS[d] for d in CRITICAL)
    if any(s == 0 for s in scores.values()):
        return False, "BLOCKED_ZERO_FACTOR", hps
    if any(value < policy["min_dimension"] for value in dims.values()):
        return False, "BLOCKED_DIMENSION_FLOOR", hps
    for factor, minimum in policy.get("critical_factor_floors", {}).items():
        if scores[factor] < minimum:
            return False, "BLOCKED_CRITICAL_FACTOR", hps
    if hps < policy["reject_below"]:
        return False, "AUTO_REJECT", hps
    if hps >= policy["direct_accept_from"]:
        return True, "AUTO_ACCEPT", hps
    review_threshold = (policy["reject_below"] + policy["direct_accept_from"]) / 2
    return (True, "REVIEW_ACCEPT", hps) if review >= review_threshold else (False, "REVIEW_REJECT", hps)


def read_archive(archive_path):
    """Load only existing decisions and original manifest, directly from ZIP."""
    pattern = re.compile(r"/adult_alpha(0\.5|0\.1)/decisions_adult_seed(4[2-6])_(clean|degraded)_none\.json$")
    by_condition = {}
    manifests = {}
    with zipfile.ZipFile(archive_path) as zf:
        paths = zf.namelist()
        for path in paths:
            m = pattern.search("/" + path.lstrip("/"))
            if m:
                key = (m.group(1), int(m.group(2)), m.group(3))
                if key in by_condition:
                    raise ValueError(f"Duplicate archived condition: {key}")
                by_condition[key] = json.loads(zf.read(path))
        for alpha in ALPHAS:
            hits = [p for p in paths if p.endswith(f"adult_alpha{alpha}/manifest_adult_K50_E7702.json")]
            if len(hits) != 1:
                raise ValueError(f"Expected one original Adult manifest for alpha={alpha}; found {len(hits)}")
            manifests[alpha] = json.loads(zf.read(hits[0]))

    expected = {(a, s, c) for a in ALPHAS for s in FL_SEEDS for c in CONDITIONS}
    if set(by_condition) != expected:
        raise ValueError(f"Missing conditions: {sorted(expected - set(by_condition))}; "
                         f"unexpected: {sorted(set(by_condition) - expected)}")
    if manifests["0.5"]["sha256"] != manifests["0.1"]["sha256"]:
        raise ValueError("Different documentary manifests across alphas; investigate before pairing")
    manifest = manifests["0.5"]
    if original_manifest_digest(manifest) != manifest["sha256"]:
        raise ValueError("Original documentary manifest hash check failed")
    return by_condition, manifest


def validate_and_prepare(archived, original_manifest):
    """Fail rather than silently changing the original experiment."""
    ids = sorted(original_manifest["clients"])
    if len(ids) != 50:
        raise ValueError(f"Expected 50 clients, found {len(ids)}")
    fixed_tiers = {cid: original_manifest["clients"][cid]["tier"] for cid in ids}
    original_docs = generate_docs(ids, BASE_EVIDENCE_SEED)
    checks = 0
    all_d2 = {}
    for key, run in sorted(archived.items()):
        if sorted(run["decisions"]) != ids:
            raise ValueError(f"Different clients for {key}")
        if run["manifest_sha256"] != original_manifest["sha256"]:
            raise ValueError(f"Original manifest hash differs for {key}")
        policy = run["policy"]
        if any(policy[k] != v for k, v in {"min_dimension": 2.5,
              "reject_below": 3.0, "direct_accept_from": 3.5}.items()):
            raise ValueError(f"Unexpected policy for {key}: {policy}")
        per_client = {}
        for cid, record in run["decisions"].items():
            archived_scores = record["factor_scores"]
            orig_scores = {f: original_manifest["clients"][cid]["documentary_factors"][f]["rubric_level"]
                           for f in DOC_FACTORS}
            if original_docs[cid]["scores"] != orig_scores:
                raise AssertionError(f"The original synthetic generator did not reproduce {cid}")
            if any(archived_scores[f] != orig_scores[f] for f in DOC_FACTORS):
                raise AssertionError(f"Documentary scores do not match manifest for {key}, {cid}")
            accepted, path, hps = evaluate_hps(archived_scores, policy)
            if accepted != record["accepted"] or path != record["path"]:
                raise AssertionError(f"Original admission decision differs for {key}, {cid}: {path}")
            if abs(hps - record["hps"]) > 1e-8:
                raise AssertionError(f"Original HPS differs for {key}, {cid}")
            per_client[cid] = {f: archived_scores[f] for f in D2_FACTORS}
            checks += 1
        all_d2[key] = per_client
    print(f"Verified {checks} original Adult client assessments and original evidence seed 7702.")
    return ids, fixed_tiers, all_d2


def write_csv(path, records, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


def mean(values):
    return statistics.mean(values)


def sample_sd(values):
    return statistics.stdev(values) if len(values) >= 2 else 0.0


def run(archive_path, output_dir, n_draws=2000, seed_start=8801):
    """Fast, exact vectorized reassessment; reuses the *original* evaluation policy."""
    import numpy as np  # Available by default in Google Colab.
    if n_draws < 2:
        raise ValueError("At least two draws are needed to calculate sample SD.")
    if BASE_EVIDENCE_SEED in range(seed_start, seed_start + n_draws):
        raise ValueError("Do not include the original evidence seed 7702.")
    archived, manifest = read_archive(archive_path)
    ids, fixed_tiers, fixed_d2 = validate_and_prepare(archived, manifest)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # Each row is one ORIGINAL client assessment; 50 clients x 5 FL seeds
    # x 2 alphas x 2 conditions = 1,000 rows.
    cases = [(a, c, s, cid) for a in ALPHAS for c in CONDITIONS
             for s in FL_SEEDS for cid in ids]
    client_idx = np.asarray([ids.index(cid) for _, _, _, cid in cases], dtype=int)
    d2_matrix = np.asarray([[fixed_d2[(a, s, c)][cid][f] for f in FACTORS['D2']]
                            for a, c, s, cid in cases], dtype=float)
    d2_mean = d2_matrix.mean(axis=1)
    d2_has_zero = np.any(d2_matrix == 0, axis=1)
    old_accept = np.asarray([archived[(a, s, c)]['decisions'][cid]['accepted']
                             for a, c, s, cid in cases], dtype=bool)
    old_hps = np.asarray([archived[(a, s, c)]['decisions'][cid]['hps']
                          for a, c, s, cid in cases], dtype=float)
    groups = {(a, c): np.asarray([i for i, (a1, c1, _, _) in enumerate(cases)
                                  if a1 == a and c1 == c], dtype=int)
              for a in ALPHAS for c in CONDITIONS}
    for key in archived:
        if archived[key]['policy'].get('critical_factor_floors'):
            raise ValueError('This fast path needs a new implementation for nonempty critical-factor floors.')

    doc_names = sorted(DOC_FACTORS)
    doc_cols = {f: j for j, f in enumerate(doc_names)}
    base_count = {key: float(np.mean(old_accept[idx]) * len(ids))
                  for key, idx in groups.items()}
    result_rows = []
    # Validate the vectorization against the original scalar policy for the FIRST draw.
    verified_fast_path = False
    for mode in ('within_tier', 'new_profiles'):
        for draw_idx, evidence_seed in enumerate(range(seed_start, seed_start + n_draws)):
            docs = generate_docs(ids, evidence_seed,
                                 profile_map=fixed_tiers if mode == 'within_tier' else None)
            doc_array = np.asarray([[docs[cid]['scores'][f] for f in doc_names]
                                    for cid in ids], dtype=float)
            doc_means = {dim: np.mean(doc_array[:, [doc_cols[f] for f in fs]], axis=1)
                         for dim, fs in FACTORS.items() if dim != 'D2'}
            zero_documentary = np.any(doc_array == 0, axis=1)
            dimension_floor_fails = d2_mean < 2.5
            for dim in doc_means:
                dimension_floor_fails = dimension_floor_fails | (doc_means[dim][client_idx] < 2.5)
            zero_fails = d2_has_zero | zero_documentary[client_idx]
            hps = WEIGHTS['D2'] * d2_mean
            for dim, dim_means in doc_means.items():
                hps = hps + WEIGHTS[dim] * dim_means[client_idx]
            review = (WEIGHTS['D1'] * doc_means['D1'][client_idx] +
                      WEIGHTS['D2'] * d2_mean +
                      WEIGHTS['D5'] * doc_means['D5'][client_idx] +
                      WEIGHTS['D6'] * doc_means['D6'][client_idx]) / 0.80
            accepted = (~zero_fails & ~dimension_floor_fails &
                        ((hps >= 3.5) | ((hps >= 3.0) & (review >= 3.25))))

            if not verified_fast_path:
                for i in range(len(cases)):
                    a, c, s, cid = cases[i]
                    all_scores = {**fixed_d2[(a, s, c)][cid], **docs[cid]['scores']}
                    expected_accept, _, expected_hps = evaluate_hps(all_scores, archived[(a, s, c)]['policy'])
                    if bool(accepted[i]) != expected_accept or abs(float(hps[i]) - expected_hps) > 1e-9:
                        raise AssertionError(f'Vectorized policy does not match original for {a}, {c}, {s}, {cid}')
                print('Verified fast calculation exactly matches the original policy across 1,000 cases.')
                verified_fast_path = True

            for alpha in ALPHAS:
                for condition in CONDITIONS:
                    idx = groups[(alpha, condition)]
                    same = (accepted[idx] == old_accept[idx])
                    result_rows.append({
                        'mode': mode, 'evidence_seed': evidence_seed,
                        'alpha': alpha, 'condition': condition,
                        'baseline_admitted_mean_over_5_seeds': round(base_count[(alpha, condition)], 5),
                        'new_admitted_mean_over_5_seeds': round(float(accepted[idx].sum()) / len(FL_SEEDS), 5),
                        'decision_agreement_pct': round(100 * float(same.mean()), 5),
                        'mean_abs_hps_difference': round(float(np.mean(np.abs(hps[idx] - old_hps[idx]))), 6),
                        'accept_to_reject_occurrences': int((old_accept[idx] & ~accepted[idx]).sum()),
                        'reject_to_accept_occurrences': int((~old_accept[idx] & accepted[idx]).sum()),
                        'clients_compared': int(len(idx)),
                    })
        print(f'Finished {n_draws:,} documentary draws in mode: {mode}')

    summary_rows = []
    for mode in ('within_tier', 'new_profiles'):
        for alpha in ALPHAS:
            for condition in CONDITIONS:
                rr = [r for r in result_rows if r['mode'] == mode and
                      r['alpha'] == alpha and r['condition'] == condition]
                counts = [r['new_admitted_mean_over_5_seeds'] for r in rr]
                agreements = [r['decision_agreement_pct'] for r in rr]
                deltas = [r['mean_abs_hps_difference'] for r in rr]
                summary_rows.append({
                    'mode': mode, 'alpha': alpha, 'condition': condition,
                    'n_documentary_draws': len(rr), 'n_fl_seeds_fixed': len(FL_SEEDS),
                    'baseline_mean_admitted': rr[0]['baseline_admitted_mean_over_5_seeds'],
                    'new_mean_admitted': round(mean(counts), 4),
                    'new_sd_across_documentary_draws': round(sample_sd(counts), 4),
                    'new_min_mean_admitted': round(min(counts), 4),
                    'new_max_mean_admitted': round(max(counts), 4),
                    'decision_agreement_mean_pct': round(mean(agreements), 2),
                    'decision_agreement_sd_pct': round(sample_sd(agreements), 2),
                    'mean_abs_hps_difference': round(mean(deltas), 5),
                    'accept_to_reject_mean_per_draw': round(mean([r['accept_to_reject_occurrences'] for r in rr]), 2),
                    'reject_to_accept_mean_per_draw': round(mean([r['reject_to_accept_occurrences'] for r in rr]), 2),
                    'min_decision_agreement_pct': round(min(agreements), 2),
                    'max_decision_agreement_pct': round(max(agreements), 2),
                })
    write_csv(out / 'evidence_stability_summary.csv', summary_rows, list(summary_rows[0]))
    write_csv(out / 'evidence_stability_by_draw.csv', result_rows, list(result_rows[0]))
    (out / 'METHODS_README.txt').write_text(
        'Adult HPS-28 synthetic documentary evidence stability (no model retraining)\n'
        f'{n_draws:,} independent evidence draws per mode; evidence seeds {seed_start}–{seed_start+n_draws-1}.\n'
        'Fifty clients; original FL seeds 42–46 fixed; alphas 0.5/0.1; clean/degraded.\n'
        'within_tier: same documentary profile tier, redraw all 20 factor values.\n'
        'new_profiles: redraw profile tier from probabilities 52/33/10/5 and all 20 values.\n'
        'Same 8 measured D2 scores, same admission policy, same original archived cases.\n'
        'Each draw compares 250 historical client assessments per alpha-condition.\n'
        'Mean admitted = average admissions per FL seed; SD = across documentary draws\n'
        'of five-seed mean admissions, NOT independent training-seed SD.\n'
        'Decision agreement = agreement with original synthetic evidence assignment.\n'
        'NOT document verification accuracy, real-document generalization, or predictive accuracy.\n'
        'Increasing draws improves precision for the assumed synthetic generator only.\n',
        encoding='utf-8',
    )
    print(f'\nSynthetic evidence sensitivity ({n_draws:,} documentary draws per mode):')
    print(f'{"MODE":<14} {"alpha":<5} {"data":<9} {"original":>9} {"resampled mean±SD":>20} {"agreement":>11}')
    for row in summary_rows:
        print(f"{row['mode']:<14} {row['alpha']:<5} {row['condition']:<9} "
              f"{row['baseline_mean_admitted']:>9.1f} "
              f"{row['new_mean_admitted']:>8.2f}±{row['new_sd_across_documentary_draws']:<8.2f} "
              f"{row['decision_agreement_mean_pct']:>10.2f}%")
    print(f'\nSaved results: {out.resolve()}')
    return summary_rows

def _resolve_archive(path):
    """Find the archived Adult results ZIP; reject raw adult.csv explicitly."""
    path = Path(path)

    def accept(candidate):
        if candidate.suffix.lower() != ".zip" or not zipfile.is_zipfile(candidate):
            raise ValueError(
                "The experiment needs the ORIGINAL Adult RESULTS ZIP, not adult.csv. "
                "Upload TADP_HPS28_Adult_Phased_Results_v5.zip "
                "(containing archived decision JSON files)."
            )
        return candidate

    for candidate in (path, Path("/content") / path.name):
        if candidate.is_file():
            return accept(candidate)

    # Common case: the archive was already uploaded with a (1) filename suffix.
    # Use it only when a single unambiguous match exists.
    candidates = []
    for folder in (Path.cwd(), Path("/content")):
        if folder.is_dir():
            candidates.extend(p for p in folder.glob("TADP_HPS28_Adult_Phased_Results_v5*.zip")
                              if p.is_file() and zipfile.is_zipfile(p))
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if len(candidates) == 1:
        print(f"Found existing Adult results ZIP: {candidates[0].name}")
        return candidates[0]

    try:
        import google.colab  # noqa: F401
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            "Adult results ZIP not found. Run with "
            "--archive /path/to/TADP_HPS28_Adult_Phased_Results_v5.zip. "
            "The raw adult.csv dataset is NOT used in this analysis."
        )

    for attempt in range(2):
        print("\nUPLOAD THIS FILE (not adult.csv):")
        print("  TADP_HPS28_Adult_Phased_Results_v5.zip")
        print("It contains prior HPS decisions and documentary scores.")
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded
                      if name.lower().endswith(".zip") and zipfile.is_zipfile(name)]
        if len(candidates) == 1:
            return accept(candidates[0])
        print("Incorrect selection: please upload the original RESULTS .zip file. ")
        print("adult.csv is the raw dataset and cannot replace the archived decisions.")
    raise FileNotFoundError(
        "No Adult results ZIP was provided. Download or locate "
        "TADP_HPS28_Adult_Phased_Results_v5.zip and run again."
    )


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--archive", type=Path, default=Path("TADP_HPS28_Adult_Phased_Results_v5.zip"),
                        help="Existing Adult phased results ZIP; no source dataset needed")
    parser.add_argument("--output", type=Path, default=Path("TADP_Adult_Evidence_Stability"))
    parser.add_argument("--draws", type=int, default=2000,
                        help="Number of independent documentary draws for EACH mode (default: 2000)")
    parser.add_argument("--seed-start", type=int, default=8801)

    # Jupyter/Google Colab injects '-f ...kernel.json' into sys.argv.
    # Ignore ONLY that notebook-internal argument. This fixes the SystemExit: 2.
    actual_argv = list(sys.argv[1:] if argv is None else argv)
    clean_argv = []
    i = 0
    while i < len(actual_argv):
        if actual_argv[i] == "-f" and i + 1 < len(actual_argv) and actual_argv[i + 1].endswith(".json"):
            i += 2
        else:
            clean_argv.append(actual_argv[i])
            i += 1

    args = parser.parse_args(clean_argv)
    archive = _resolve_archive(args.archive)
    summary = run(archive, args.output, args.draws, args.seed_start)
    return summary


if __name__ == "__main__":
    main()


Verified 1000 original Adult client assessments and original evidence seed 7702.
Verified fast calculation exactly matches the original policy across 1,000 cases.
Finished 2,000 documentary draws in mode: within_tier
Finished 2,000 documentary draws in mode: new_profiles

Synthetic evidence sensitivity (2,000 documentary draws per mode):
MODE           alpha data       original    resampled mean±SD   agreement
within_tier    0.5   clean          30.2    30.20±0.02         100.00%
within_tier    0.5   degraded       29.4    29.54±0.14          99.49%
within_tier    0.1   clean          16.2    16.40±0.03          99.60%
within_tier    0.1   degraded       14.6    14.78±0.08          99.56%
new_profiles   0.5   clean          30.2    30.96±1.90          80.39%
new_profiles   0.5   degraded       29.4    30.23±1.86          80.58%
new_profiles   0.1   clean          16.2    16.32±1.11          90.00%
new_profiles   0.1   degraded       14.6    14.75±1.02          90.77%

Saved results: /c

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""TADP-HPS28: AUTOMATIC simulated HUMAN reviewer +/-1 rubric uncertainty.

ONE-RUN, ZERO PROMPTS, NO AI / API, NO HUMAN INPUT.

Creates 1,000 independent synthetic documentary assessment profiles (20 Appendix A
rubric descriptors per profile). For D2 it preferentially REPLAYS the archived
TRAIN-only eight-factor scores from the user's original Adult benchmark (1,000
client-condition-seed assessments = 50 virtual clients x 2 alphas x 2 data
conditions x 5 seeds). If the ZIP cannot be found, a clearly marked synthetic
D2 *score* fallback is used. No raw data, real evidence documents, or human
reviewers are processed or impersonated.

For EVERY synthetic assessment, changes ONE of the 20 documentary levels by
exactly +1 or -1 (within 0...5), holding the remaining 27 factors fixed, and
recalculates the exact TADP-HPS-28/1.0 decision. Evaluates all feasible shifts
(exhaustive, not a stochastic estimate of human error rate), with special
reporting on 0->1 (unsupported evidence incorrectly accepted) and 1->0.

All outputs are descriptive and conditional on a hypothetical single-factor
review error. A flip rate is NOT an empirical probability of human error.

RUN IN COLAB: upload file then run:
    %run TADP_Reviewer_PM1_Sensitivity_AUTO.py
or in terminal: python TADP_Reviewer_PM1_Sensitivity_AUTO.py
No arguments, input prompts or API credentials are needed.

Dependencies: Python 3.10+, matplotlib (available in Colab).
"""
from __future__ import annotations

import csv
import json
import math
import random
import re
import statistics
import sys
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

N_PROFILES = 1000
EVIDENCE_SEED = 7702
ERROR_SEED = 20261108
RESULTS_DIR = Path('TADP_Reviewer_PM1_Results')
RESULTS_ZIP = Path('TADP_Reviewer_PM1_Sensitivity_RESULTS.zip')

POLICY_VERSION = 'TADP-HPS-28/1.0'
MIN_DIM = 2.5
REJECT_BELOW = 3.0
DIRECT_ACCEPT_FROM = 3.5
REVIEW_MIDPOINT = (REJECT_BELOW + DIRECT_ACCEPT_FROM) / 2

FACTORS = {
 'D1': ('source_reputation', 'data_controller', 'data_objective', 'data_collection_lineage'),
 'D2': ('completeness', 'duplication_rate', 'value_validity_error_rate', 'type_consistency',
        'label_integrity', 'feature_distribution_consistency', 'feature_category_coverage',
        'structural_constraint_integrity'),
 'D3': ('data_dictionary', 'version_logs', 'collection_protocol', 'definition_updates'),
 'D4': ('data_freshness', 'scheduled_refresh', 'retention_clarity'),
 'D5': ('regulation_coverage', 'consent_ethics', 'geo_restrictions', 'sensitivity_classification', 'audits'),
 'D6': ('license_terms', 'ethical_reviews', 'redistribution', 'user_agreements'),
}
WEIGHTS = {'D1': 0.25, 'D2': 0.15, 'D3': 0.10, 'D4': 0.10, 'D5': 0.30, 'D6': 0.10}
CRITICAL = ('D1', 'D2', 'D5', 'D6')
DOC_FACTORS = sorted(f for d, factors in FACTORS.items() if d != 'D2' for f in factors)
D2_FACTORS = FACTORS['D2']
ALL_FACTORS = set(DOC_FACTORS) | set(D2_FACTORS)
DIM_BY_FACTOR = {f: d for d, fs in FACTORS.items() for f in fs}
assert len(DOC_FACTORS) == 20 and len(ALL_FACTORS) == 28 and abs(sum(WEIGHTS.values())-1)<1e-12

# Exact user-supplied Appendix A documentary descriptors; order = deterministic level 0..5.
RUBRICS = {
 'source_reputation': ('No information','Poor','Limited evidence','Average, partially trusted','Well-documented, reliable','Highly reputable, verified'),
 'data_controller': ('No documented controller','Unclear','Partially clear','Moderately clear','Mostly clear','Fully documented'),
 'data_objective': ('None','Vague','Partial','General but unclear','Mostly explicit','Fully explicit and justified'),
 'data_collection_lineage': ('Origin unknown','Informal/unverifiable origin','Partially documented path','Documented with limited traceability','Well-documented traceable path','Fully source-linked/versioned/auditable'),
 'data_dictionary': ('None','Minimal outline','Partial coverage','Moderate coverage','Near-complete','Fully detailed'),
 'version_logs': ('None','Minimal logs','Occasional logs','Regular logs','Near-complete','Full version history'),
 'collection_protocol': ('None','Vague','Partial','General methods','Well-defined','Fully transparent'),
 'definition_updates': ('None','Rare','Occasional','Regular but basic','Frequent','Real-time documented'),
 'data_freshness': ('>5 years','2-5 years','1-2 years','6-12 months','1-6 months','Current'),
 'scheduled_refresh': ('Never','Irregular','Annual','Quarterly','Monthly','Daily/real time'),
 'retention_clarity': ('None','Minimal','Basic guidelines','Moderate clarity','High clarity','Fully documented'),
 'regulation_coverage': ('None','Minimal','Partial','Moderate','Comprehensive but dated','Fully documented/current'),
 'consent_ethics': ('None','Minimal record','Partial','Moderate logs','Substantial','Fully documented'),
 'geo_restrictions': ('None','Basic mention','Partial','Moderate','Near-complete','Fully documented'),
 'sensitivity_classification': ('None','Basic flagging','Partial','Moderate','Near-complete','Fully classified'),
 'audits': ('None','Internal only','Basic certification','Occasional audit','Recent audit','Regular external audits'),
 'license_terms': ('None','Vague','Basic','Clear','Detailed','Industry-compliant'),
 'ethical_reviews': ('None','Informal','Partial','Moderate','Well-documented','Certified'),
 'redistribution': ('None','Unclear','Partial','Clear','Detailed','Fully documented'),
 'user_agreements': ('Non-compliant','Minimal','Partial','Mostly compliant','Fully compliant','Audited'),
}
assert set(RUBRICS)==set(DOC_FACTORS) and all(len(x)==6 for x in RUBRICS.values())

BANDS = {'high':(4,5),'medium':(3,4),'low':(1,2),'missing':(0,1)}
BAND_PROBABILITIES = (.52, .33, .10, .05)


def evaluate(scores):
    """Faithful reproduction of original non-ablation TADP 28-factor admission policy."""
    if set(scores) != ALL_FACTORS:
        raise ValueError('Missing/unexpected factor identifier')
    if any(type(v)!=int or not 0 <= v <= 5 for v in scores.values()):
        raise ValueError('Every factor must have an integer rubric level 0..5')
    d = {dim: sum(scores[f] for f in fs)/len(fs) for dim, fs in FACTORS.items()}
    hps = sum(WEIGHTS[dim]*d[dim] for dim in FACTORS)
    review_score = sum(WEIGHTS[dim]*d[dim] for dim in CRITICAL)/sum(WEIGHTS[dim] for dim in CRITICAL)
    if any(v==0 for v in scores.values()):
        accepted, path = False, 'BLOCKED_ZERO_FACTOR'
    elif any(x < MIN_DIM for x in d.values()):
        accepted, path = False, 'BLOCKED_DIMENSION_FLOOR'
    elif hps < REJECT_BELOW:
        accepted, path = False, 'AUTO_REJECT'
    elif hps >= DIRECT_ACCEPT_FROM:
        accepted, path = True, 'AUTO_ACCEPT'
    elif review_score >= REVIEW_MIDPOINT:
        accepted, path = True, 'REVIEW_ACCEPT'
    else:
        accepted, path = False, 'REVIEW_REJECT'
    return {'accepted':accepted, 'path':path, 'hps':round(hps,10),
            'review_score':round(review_score,10), 'dimensions':d}


def read_adult_archive():
    """Optionally reuse REAL computational D2 from user's archived TRAIN-only simulation.

    These are repeated assessments of 50 *simulated* clients, NOT 1,000 distinct
    verified providers. Reused only as plausible measured D2 backgrounds.
    """
    names = ('TADP_HPS28_Adult_Phased_Results_v5.zip',
             '/content/TADP_HPS28_Adult_Phased_Results_v5.zip',
             '/mnt/data/TADP_HPS28_Adult_Phased_Results_v5.zip')
    archive = next((Path(x) for x in names if Path(x).is_file()),None)
    if archive is None:
        return [], {'source':'synthetic_D2_fallback', 'archive_found':False,
                    'archived_reproduction_checked':False}
    pattern = re.compile(r'adult_alpha(0\.5|0\.1)/decisions_adult_seed(4[2-6])_(clean|degraded)_none\.json$')
    records=[]
    checks=0
    with zipfile.ZipFile(archive) as z:
        for name in sorted(z.namelist()):
            m=pattern.search(name)
            if not m: continue
            payload=json.loads(z.read(name))
            policy=payload.get('policy',{})
            for key, expected in (('min_dimension',MIN_DIM),('reject_below',REJECT_BELOW),
                                  ('direct_accept_from', DIRECT_ACCEPT_FROM)):
                if policy.get(key)!=expected: raise ValueError(f'Archive policy mismatch {name} {key}')
            if policy.get('critical_factor_floors'):
                raise ValueError('Cannot ignore archived critical factor floors')
            for cid, row in sorted(payload['decisions'].items()):
                d2={f:row['factor_scores'][f] for f in D2_FACTORS}
                base=evaluate({f:row['factor_scores'][f] for f in ALL_FACTORS})
                if (base['accepted']!=row['accepted'] or base['path']!=row['path'] or
                    abs(base['hps']-row['hps'])>1e-8):
                    raise AssertionError(f'Archived policy did not reproduce {name} {cid}')
                checks+=1
                records.append({'original_id':cid, 'alpha':m.group(1),
                                'condition':m.group(3), 'fl_seed':int(m.group(2)),
                                'd2':d2})
    if len(records)!=1000 or checks!=1000:
        raise ValueError(f'Expected 1,000 archived assessments; got {len(records)}')
    return records, {'source':'archived_Adult_TRAIN_D2', 'archive_found':True,
                     'archive_file':archive.name, 'archived_reproduction_checked':True,
                     'original_admission_matches':checks,
                     'source_caveat':'D2 reused from 50 simulated clients x 2 alpha x 2 conditions x 5 seeds; not 1000 independent TRAIN partitions.'}


def synthetic_d2(rng):
    """Explicit FALLBACK: synthetic *scores*, not real raw-TRAIN D2 calculations."""
    return {f: rng.choices(range(6), weights=(2,4,9,15,30,40), k=1)[0]
            if f in ('feature_distribution_consistency','feature_category_coverage')
            else rng.choices(range(6), weights=(1,1,2,4,20,72), k=1)[0]
            for f in D2_FACTORS}


def create_simulated_cases(archived):
    rng=random.Random(EVIDENCE_SEED)
    cases=[]
    for i in range(N_PROFILES):
        band=rng.choices(tuple(BANDS),weights=BAND_PROBABILITIES,k=1)[0]
        lo,hi=BANDS[band]
        # Each distinct case gets twenty independently selected *synthetic reviewer levels*
        # conditioned on the same documentary band, like the original benchmark.
        docs={f:rng.randint(lo,hi) for f in DOC_FACTORS}
        if archived:
            source=archived[i]  # 1,000 historical D2 assessment contexts, one per case
            d2=source['d2']
            origin={k:source[k] for k in ('original_id','alpha','condition','fl_seed')}
        else:
            d2=synthetic_d2(rng)
            origin={'original_id':'', 'alpha':'synthetic',
                    'condition':'synthetic', 'fl_seed':''}
        complete={**docs,**d2}
        baseline=evaluate(complete)
        cases.append({'id':f'SIM-{i+1:04d}','tier':band,'docs':docs,'d2':d2,
                      'origin':origin,'scores':complete,'baseline':baseline})
    return cases


def write_csv(path, rows, fieldnames):
    with Path(path).open('w',newline='',encoding='utf8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames,extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)


def pct(n,d):return 100*n/d if d else 0


def report_and_outputs(cases, source_info):
    RESULTS_DIR.mkdir(parents=True,exist_ok=True)
    library=[]
    baseline_rows=[]
    scenario_rows=[]
    factor_stats=defaultdict(lambda: Counter())
    dim_stats=defaultdict(lambda: Counter())
    error_type_stats=defaultdict(lambda:Counter())
    baseline_counts=Counter()
    case_flips=defaultdict(set)
    scenario_per_case=Counter()
    factor_effects={f:WEIGHTS[DIM_BY_FACTOR[f]]/len(FACTORS[DIM_BY_FACTOR[f]]) for f in DOC_FACTORS}

    for case in cases:
        b=case['baseline']
        baseline_counts[(case['tier'],'accept' if b['accepted'] else 'reject')]+=1
        library_row={'synthetic_assessment_id':case['id'], 'profile_band':case['tier'],
                     'authentic_uploaded_documents':False,
                     'reviewer':'SIMULATED_NOT_HUMAN',
                     'profile_seed':EVIDENCE_SEED}
        for f in DOC_FACTORS:
            library_row[f+'_level']=case['docs'][f]
            library_row[f+'_descriptor']=RUBRICS[f][case['docs'][f]]
        library.append(library_row)
        baseline_row={'assessment_id':case['id'],'band':case['tier'],
                      **case['origin'],
                      'baseline_accepted':int(b['accepted']),
                      'baseline_path':b['path'], 'baseline_hps':b['hps'],
                      'baseline_review_score':b['review_score']}
        baseline_row.update({d+'_mean':round(b['dimensions'][d],6) for d in FACTORS})
        baseline_row.update({'D2_'+f:case['d2'][f] for f in D2_FACTORS})
        baseline_rows.append(baseline_row)
        for f in DOC_FACTORS:
            old=case['docs'][f]
            for delta in (-1,1):
                new=old+delta
                if not 0<=new<=5:continue
                altered=case['scores'].copy()
                altered[f]=new
                v=evaluate(altered)
                flipped=(v['accepted']!=b['accepted'])
                direction=('reject_to_accept' if (flipped and v['accepted']) else
                           'accept_to_reject' if flipped else 'no_change')
                evidence_case=('unsupported_0_to_1' if old==0 and delta==1 else
                               'supported_1_to_0' if old==1 and delta==-1 else
                               'established_levels_only')
                row={'assessment_id':case['id'],'profile_band':case['tier'],
                     'factor':f,'dimension':DIM_BY_FACTOR[f],
                     'change':delta,'original_level':old,'perturbed_level':new,
                     'original_descriptor':RUBRICS[f][old],
                     'perturbed_descriptor':RUBRICS[f][new],
                     'evidence_boundary_type':evidence_case,
                     'baseline_accepted':int(b['accepted']),
                     'perturbed_accepted':int(v['accepted']),
                     'baseline_path':b['path'],'perturbed_path':v['path'],
                     'baseline_hps':b['hps'],'perturbed_hps':v['hps'],
                     'delta_hps':round(v['hps']-b['hps'],10),
                     'admission_changed':int(flipped),'transition':direction,
                     'original_D2_source':source_info['source']}
                scenario_rows.append(row)
                scenario_per_case[case['id']]+=1
                if flipped:case_flips[case['id']].add(f)
                for counts in (factor_stats[f],dim_stats[DIM_BY_FACTOR[f]],error_type_stats[evidence_case]):
                    counts['tested']+=1
                    counts['changed']+=int(flipped)
                    counts[direction]+=1
                    counts['baseline_accepted']+=int(b['accepted'])
                    counts['baseline_rejected']+=int(not b['accepted'])
                    counts['upward_tests']+=int(delta==1)
                    counts['downward_tests']+=int(delta==-1)
                    counts['upward_changes']+=int(flipped and delta==1)
                    counts['downward_changes']+=int(flipped and delta==-1)
                # If a single-factor mismatch crosses accept/reject, record the case at-risk.

    total=len(scenario_rows)
    n_flips=sum(r['admission_changed'] for r in scenario_rows)
    ra=sum(r['transition']=='reject_to_accept' for r in scenario_rows)
    ar=sum(r['transition']=='accept_to_reject' for r in scenario_rows)
    orig_accept=sum(c['baseline']['accepted'] for c in cases)

    # Factual mathematical property: a +/- 1 score always changes HPS by +/- weight / # factors.
    for f in DOC_FACTORS:
        related=[r for r in scenario_rows if r['factor']==f]
        if related and any(abs(abs(r['delta_hps'])-factor_effects[f])>1e-8 for r in related):
            raise AssertionError(f'Unexpected HPS score shift for {f}')

    factor_rows=[]
    for f in DOC_FACTORS:
        c=factor_stats[f]
        factor_rows.append({'factor':f,'dimension':DIM_BY_FACTOR[f],
                            'weight_per_level':round(factor_effects[f],9),
                            'tested_single_errors':c['tested'],
                            'admission_flips':c['changed'],
                            'flip_pct_of_feasible_shifts':round(pct(c['changed'],c['tested']),4),
                            'reject_to_accept':c['reject_to_accept'],
                            'accept_to_reject':c['accept_to_reject'],
                            'upward_shifts':c['upward_tests'],
                            'upward_flips':c['upward_changes'],
                            'downward_shifts':c['downward_tests'],
                            'downward_flips':c['downward_changes']})
    factor_rows.sort(key=lambda x:(-x['flip_pct_of_feasible_shifts'],-x['admission_flips'],x['factor']))

    dimension_rows=[]
    for d in ('D1','D3','D4','D5','D6'):
        c=dim_stats[d]
        dimension_rows.append({'dimension':d,'documentary_factors':len(FACTORS[d]),
                               'dimension_weight':WEIGHTS[d],
                               'tested_single_errors':c['tested'],
                               'admission_flips':c['changed'],
                               'flip_pct_of_feasible_shifts':round(pct(c['changed'],c['tested']),4),
                               'reject_to_accept':c['reject_to_accept'],
                               'accept_to_reject':c['accept_to_reject']})

    errors_by_type=[]
    for t,c in sorted(error_type_stats.items()):
        errors_by_type.append({'error_type':t,'tested_single_errors':c['tested'],
                                'admission_flips':c['changed'],
                                'flip_pct_of_feasible_shifts':round(pct(c['changed'],c['tested']),4),
                                'reject_to_accept':c['reject_to_accept'],
                                'accept_to_reject':c['accept_to_reject']})
    protected=[r for r in scenario_rows if r['evidence_boundary_type']=='established_levels_only']
    protected_flips=sum(r['admission_changed'] for r in protected)
    harmful_01=[r for r in scenario_rows if r['evidence_boundary_type']=='unsupported_0_to_1']

    # Distinguish "one error anywhere" vs the hypothetical "already-positive evidence only" scenario.
    near_threshold=[c for c in cases if (abs(c['baseline']['hps']-3.0)<=0.1 or
                                        abs(c['baseline']['hps']-3.5)<=0.1)]
    cohort_rows=[]
    for tier in BANDS:
        subset=[c for c in cases if c['tier']==tier]
        related=[r for r in scenario_rows if r['profile_band']==tier]
        cohort_rows.append({'profile_band':tier,'cases':len(subset),
                            'baseline_admitted':sum(c['baseline']['accepted'] for c in subset),
                            'shifts_tested':len(related),
                            'admission_flips':sum(r['admission_changed'] for r in related),
                            'flip_pct_of_shifts':round(pct(sum(r['admission_changed'] for r in related),len(related)),4)})
    num_vulnerable=len(case_flips)
    summary={
        'experiment':'Reviewer one-factor ±1-level rubric-error sensitivity',
        'policy_version':POLICY_VERSION,
        'experiment_type':'fully_synthetic_documentary_assessments',
        'simulated_independent_documentary_profiles':len(cases),
        'documentary_factors':len(DOC_FACTORS),
        'D2_factors_fixed_per_assessment':len(D2_FACTORS),
        'D2_source':source_info,
        'actual_human_reviewers':0,'authentic_documentary_evidence_packages':0,
        'repetitions_of_federated_training':0,
        'original_FL_experiment_count_unchanged':True,
        'seed_documentary':EVIDENCE_SEED,'seed_misc':ERROR_SEED,
        'tiers':{'high':.52,'medium':.33,'low':.10,'missing':.05},
        'baseline_admitted':orig_accept,
        'baseline_rejected':len(cases)-orig_accept,
        'baseline_admission_pct':round(pct(orig_accept,len(cases)),4),
        'feasible_single_factor_1_level_errors':total,
        'single_error_admission_flips':n_flips,
        'single_error_flip_pct_of_feasible_shifts':round(pct(n_flips,total),4),
        'single_error_reject_to_accept':ra,
        'single_error_accept_to_reject':ar,
        'single_error_reject_to_accept_pct_of_shifts':round(pct(ra,total),4),
        'single_error_accept_to_reject_pct_of_shifts':round(pct(ar,total),4),
        'cases_sensitive_to_at_least_one_single_factor_error':num_vulnerable,
        'pct_cases_with_any_single_factor_flip':round(pct(num_vulnerable,len(cases)),4),
        'already_supported_levels_shifts':len(protected),
        'already_supported_levels_admission_flips':protected_flips,
        'already_supported_levels_flip_pct':round(pct(protected_flips,len(protected)),4),
        'zero_to_one_false_support_shifts':len(harmful_01),
        'zero_to_one_false_support_admission_flips':sum(r['admission_changed'] for r in harmful_01),
        'cases_with_hps_within_0_1_of_any_threshold':len(near_threshold),
        'score_delta_per_factor':{f:round(factor_effects[f],9) for f in DOC_FACTORS},
        'unit_of_analysis_note':'The 1,000 synthetic documentary profiles are independent artificial cases; archived D2 contexts, when available, are reused from 50 virtual Adult clients and are not independent authentic providers.',
        'interpretation_limit':'Conditional decision robustness under a single simulated reviewer-level error; NOT measured human error rates, reviewer agreement, documentary authenticity, predictive performance, or proof of trustworthy input data.'
    }

    write_csv(RESULTS_DIR/'SIMULATED_Documentary_Library_1000.csv',library,list(library[0]))
    write_csv(RESULTS_DIR/'Baseline_1000.csv',baseline_rows,list(baseline_rows[0]))
    write_csv(RESULTS_DIR/'Exhaustive_One_Factor_PM1.csv',scenario_rows,list(scenario_rows[0]))
    write_csv(RESULTS_DIR/'Factor_Impact.csv',factor_rows,list(factor_rows[0]))
    write_csv(RESULTS_DIR/'Dimension_Impact.csv',dimension_rows,list(dimension_rows[0]))
    write_csv(RESULTS_DIR/'Evidence_Boundary_Impact.csv',errors_by_type,list(errors_by_type[0]))
    write_csv(RESULTS_DIR/'Profile_Band_Impact.csv',cohort_rows,list(cohort_rows[0]))
    (RESULTS_DIR/'Summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n',encoding='utf8')

    # Automatically produce two publication-oriented figures, separate axes and no subplots.
    try:
        import matplotlib.pyplot as plt
        sorted_chart=sorted(factor_rows,key=lambda r:r['flip_pct_of_feasible_shifts'])
        fig,ax=plt.subplots(figsize=(10,7.5))
        ax.barh([r['factor'] for r in sorted_chart],[r['flip_pct_of_feasible_shifts'] for r in sorted_chart])
        ax.set_xlabel('Admission changes / feasible single-factor ±1 changes (%)')
        ax.set_title('TADP documentary assessment uncertainty by factor (synthetic)')
        ax.grid(axis='x',alpha=.15)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR/'Figure_Factor_Admission_Impact.png',dpi=180)
        plt.close(fig)

        labels=['Unchanged','Rejected → admitted','Admitted → rejected']
        vals=[total-n_flips,ra,ar]
        fig,ax=plt.subplots(figsize=(8,4.5))
        ax.bar(labels,vals)
        ax.set_ylabel('Number of feasible single-factor ±1 scenarios')
        ax.set_title(f'TADP admission robustness across {total:,} simulated reviewer errors')
        ax.tick_params(axis='x',rotation=12)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR/'Figure_Admission_Transitions.png',dpi=180)
        plt.close(fig)
    except ImportError:
        print('Matplotlib not installed; CSV and JSON outputs still generated.')

    leaders=', '.join(f"{r['factor']} ({r['flip_pct_of_feasible_shifts']:.2f}%)" for r in factor_rows[:4])
    readme=f'''# TADP reviewer ±1 rubric-error sensitivity (AUTOMATIC SYNTHETIC SIMULATION)

## Main outcomes (ONE hypothetical factor-level error per assessment)

- Independently generated synthetic documentary profiles: **{len(cases):,}** (20 factors each).
- Real uploaded documents / actual authorized reviewers: **0 / 0**.
- D2 background: **{source_info['source']}**. D2 frozen for every perturbation.
- Baseline admitted: **{orig_accept:,} / {len(cases):,}** ({pct(orig_accept,len(cases)):.2f}%).
- Exhaustive feasible one-factor ±1 rubric-level changes: **{total:,}**.
- Admission flips: **{n_flips:,}** ({pct(n_flips,total):.3f}% of feasible shifts).
- Rejected→admitted: **{ra:,}**; admitted→rejected: **{ar:,}**.
- Synthetic cases where *at least one* documentary ±1 causes a flip: **{num_vulnerable:,} / {len(cases):,}** ({pct(num_vulnerable,len(cases)):.2f}%).
- Changes among *already-supported levels* (not 0→1 / 1→0): **{protected_flips:,} / {len(protected):,}** ({pct(protected_flips,len(protected)):.3f}%).
- False support from level 0→1: **{sum(r['admission_changed'] for r in harmful_01):,} / {len(harmful_01):,}** perturbations changed admission.
- Most decision-sensitive factors by equal-weighted feasible shift frequency: {leaders}.

## Design

1. Create 1,000 **synthetic reviewer-outcome profiles** with band probabilities 52% high (4–5), 33% medium (3–4), 10% low (1–2), and 5% missing (0–1). These levels are mapped to the **exact Appendix A descriptors**; this is not a library of authentic documentary files.
2. D2: when available, reuse eight original Adult TRAIN-only D2 scores from each of the 1,000 archived client-condition-seed assessments. The archive is itself 50 virtual clients x 2 alphas x 2 clean/degraded x 5 seeds; thus its D2 backgrounds are **not 1,000 independent providers**. Otherwise use clearly marked synthetic D2 score distributions.
3. Apply original TADP-HPS28/1.0 admission: zeros fail, dimension floor 2.5, HPS reject<3, direct accept>=3.5, intermediate critical weighted review >=3.25; no extra critical factor floors or disqualifiers.
4. For every documentary factor of every case, make one **single** valid -1 and/or +1 error. Only the selected factor changes, remaining 27 are fixed. Recalculate admission. Changes at 0→1 and 1→0 are shown separately.
5. Enumerate **all feasible changes** (not a Monte Carlo estimate of mistake rates). Counts are hypothetical scenario frequencies. No human review error probability is assumed; probabilities are conditional on uniform weighting of the listed feasible shifts, not calibrated to human behavior.

## Interpreting the outcome responsibly

A low flip percentage means the chosen deterministic admission policy resists one-step perturbations *for the simulated baseline distribution*. It DOES NOT show that reviewers agree, evidence is genuine, inputs comply with laws, or the governance scheme is universally robust. A case can flip at the hard 0-factor rule even if HPS changes only slightly. Human errors may be correlated, larger than one step, or biased toward certain factors. Values are descriptive and cannot be generalized to real institutions.

## Outputs

- `SIMULATED_Documentary_Library_1000.csv`: 20 Appendix A descriptors and levels for each synthetic case, explicit NOT HUMAN flag.
- `Baseline_1000.csv`: baseline HPS and admissions, frozen D2.
- `Exhaustive_One_Factor_PM1.csv`: every feasible one-factor +/-1 alternative and admission transition.
- `Factor_Impact.csv`: per documentary factor sensitivity and reject/accept direction.
- `Dimension_Impact.csv`: aggregated dimension sensitivity.
- `Evidence_Boundary_Impact.csv`: isolate level0→1, level1→0 vs already-positive transitions.
- `Profile_Band_Impact.csv`: group-level stratification.
- `Summary.json`: quantitative summary and reproducibility settings.
- `Figure_Factor_Admission_Impact.png`, `Figure_Admission_Transitions.png`: graphics if matplotlib is installed.

## Reproducibility

Run: `%run TADP_Reviewer_PM1_Sensitivity_AUTO.py` in Google Colab. **No input or menu.** ZIP auto-detected if present. Documentary RNG seed = {EVIDENCE_SEED}; simulated profiles = {N_PROFILES}. No FL model retraining, and the existing 590 original training evaluations are untouched.
'''
    (RESULTS_DIR/'REPORT.md').write_text(readme,encoding='utf8')

    with zipfile.ZipFile(RESULTS_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(RESULTS_DIR.iterdir()):
            if p.is_file():z.write(p,arcname=f'{RESULTS_DIR.name}/{p.name}')
    print('\\n'+'='*78)
    print('AUTOMATIC TADP DOCUMENTARY HUMAN-REVIEWER +/-1 ERROR SENSITIVITY')
    print('='*78)
    print('Documentary cases:           ',f'{len(cases):,}', '(SYNTHETIC; 0 actual documents / reviewers)')
    print('D2 source:                   ',source_info['source'])
    print('Original archive verified:   ',source_info.get('original_admission_matches',0),'of 1,000')
    print('Baseline admitted:           ',f'{orig_accept:,} / {len(cases):,} ({pct(orig_accept,len(cases)):.2f}%)')
    print('All feasible +/-1 scenarios: ',f'{total:,}')
    print('Admission decisions changed: ',f'{n_flips:,} ({pct(n_flips,total):.3f}%)')
    print('  rejected -> admitted:      ',f'{ra:,}')
    print('  admitted -> rejected:      ',f'{ar:,}')
    print('Cases sensitive to >=1 error:',f'{num_vulnerable:,} / {len(cases):,} ({pct(num_vulnerable,len(cases)):.2f}%)')
    print('No-missing-evidence category:',f'{protected_flips:,} / {len(protected):,} changed ({pct(protected_flips,len(protected)):.3f}%)')
    print('Most sensitive factors:      ',leaders)
    print('RESULTS ZIP:                 ',RESULTS_ZIP.resolve())
    print('REPORT:                      ',(RESULTS_DIR/'REPORT.md').resolve())
    print('NOTE: Scenario frequencies are NOT estimates of human reviewer error rates.')
    print('='*78)
    return summary


def main():
    archive,source_info=read_adult_archive()
    cases=create_simulated_cases(archive)
    return report_and_outputs(cases,source_info)


if __name__=='__main__':
    main()


\n==============================================================================
AUTOMATIC TADP DOCUMENTARY HUMAN-REVIEWER +/-1 ERROR SENSITIVITY
Documentary cases:            1,000 (SYNTHETIC; 0 actual documents / reviewers)
D2 source:                    archived_Adult_TRAIN_D2
Original archive verified:    1000 of 1,000
Baseline admitted:            462 / 1,000 (46.20%)
All feasible +/-1 scenarios:  34,400
Admission decisions changed:  53 (0.154%)
  rejected -> admitted:       13
  admitted -> rejected:       40
Cases sensitive to >=1 error: 5 / 1,000 (0.50%)
No-missing-evidence category: 53 / 32,377 changed (0.164%)
Most sensitive factors:       consent_ethics (0.29%), data_collection_lineage (0.29%), geo_restrictions (0.29%), source_reputation (0.29%)
RESULTS ZIP:                  /content/TADP_Reviewer_PM1_Sensitivity_RESULTS.zip
REPORT:                       /content/TADP_Reviewer_PM1_Results/REPORT.md
NOTE: Scenario frequencies are NOT estimates of human reviewer error rates.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""TADP-HPS28: AUTOMATIC simulated HUMAN reviewer +/-1 rubric uncertainty.

ONE-RUN, ZERO PROMPTS, NO AI / API, NO HUMAN INPUT.

Creates 1,000 independent synthetic documentary assessment profiles (20 Appendix A
rubric descriptors per profile). For D2 it preferentially REPLAYS the archived
TRAIN-only eight-factor scores from the user's original Adult benchmark (1,000
client-condition-seed assessments = 50 virtual clients x 2 alphas x 2 data
conditions x 5 seeds). If the ZIP cannot be found, a clearly marked synthetic
D2 *score* fallback is used. No raw data, real evidence documents, or human
reviewers are processed or impersonated.

For EVERY synthetic assessment, changes ONE of the 20 documentary levels by
exactly +1 or -1 (within 0...5), holding the remaining 27 factors fixed, and
recalculates the exact TADP-HPS-28/1.0 decision. Evaluates all feasible shifts
(exhaustive, not a stochastic estimate of human error rate), with special
reporting on 0->1 (unsupported evidence incorrectly accepted) and 1->0.

All outputs are descriptive and conditional on a hypothetical single-factor
review error. A flip rate is NOT an empirical probability of human error.

RUN IN COLAB: upload file then run:
    %run TADP_Reviewer_PM1_Sensitivity_AUTO.py
or in terminal: python TADP_Reviewer_PM1_Sensitivity_AUTO.py
No arguments, input prompts or API credentials are needed.

Dependencies: Python 3.10+, matplotlib (available in Colab).
"""
from __future__ import annotations

import csv
import json
import math
import random
import re
import statistics
import sys
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

N_PROFILES = 1000
EVIDENCE_SEED = 7702
ERROR_SEED = 20261108
RESULTS_DIR = Path('TADP_Reviewer_PM1_Results')
RESULTS_ZIP = Path('TADP_Reviewer_PM1_Sensitivity_RESULTS.zip')

POLICY_VERSION = 'TADP-HPS-28/1.0'
MIN_DIM = 2.5
REJECT_BELOW = 3.0
DIRECT_ACCEPT_FROM = 3.5
REVIEW_MIDPOINT = (REJECT_BELOW + DIRECT_ACCEPT_FROM) / 2

FACTORS = {
 'D1': ('source_reputation', 'data_controller', 'data_objective', 'data_collection_lineage'),
 'D2': ('completeness', 'duplication_rate', 'value_validity_error_rate', 'type_consistency',
        'label_integrity', 'feature_distribution_consistency', 'feature_category_coverage',
        'structural_constraint_integrity'),
 'D3': ('data_dictionary', 'version_logs', 'collection_protocol', 'definition_updates'),
 'D4': ('data_freshness', 'scheduled_refresh', 'retention_clarity'),
 'D5': ('regulation_coverage', 'consent_ethics', 'geo_restrictions', 'sensitivity_classification', 'audits'),
 'D6': ('license_terms', 'ethical_reviews', 'redistribution', 'user_agreements'),
}
WEIGHTS = {'D1': 0.25, 'D2': 0.15, 'D3': 0.10, 'D4': 0.10, 'D5': 0.30, 'D6': 0.10}
CRITICAL = ('D1', 'D2', 'D5', 'D6')
DOC_FACTORS = sorted(f for d, factors in FACTORS.items() if d != 'D2' for f in factors)
D2_FACTORS = FACTORS['D2']
ALL_FACTORS = set(DOC_FACTORS) | set(D2_FACTORS)
DIM_BY_FACTOR = {f: d for d, fs in FACTORS.items() for f in fs}
assert len(DOC_FACTORS) == 20 and len(ALL_FACTORS) == 28 and abs(sum(WEIGHTS.values())-1)<1e-12

# Exact user-supplied Appendix A documentary descriptors; order = deterministic level 0..5.
RUBRICS = {
 'source_reputation': ('No information','Poor','Limited evidence','Average, partially trusted','Well-documented, reliable','Highly reputable, verified'),
 'data_controller': ('No documented controller','Unclear','Partially clear','Moderately clear','Mostly clear','Fully documented'),
 'data_objective': ('None','Vague','Partial','General but unclear','Mostly explicit','Fully explicit and justified'),
 'data_collection_lineage': ('Origin unknown','Informal/unverifiable origin','Partially documented path','Documented with limited traceability','Well-documented traceable path','Fully source-linked/versioned/auditable'),
 'data_dictionary': ('None','Minimal outline','Partial coverage','Moderate coverage','Near-complete','Fully detailed'),
 'version_logs': ('None','Minimal logs','Occasional logs','Regular logs','Near-complete','Full version history'),
 'collection_protocol': ('None','Vague','Partial','General methods','Well-defined','Fully transparent'),
 'definition_updates': ('None','Rare','Occasional','Regular but basic','Frequent','Real-time documented'),
 'data_freshness': ('>5 years','2-5 years','1-2 years','6-12 months','1-6 months','Current'),
 'scheduled_refresh': ('Never','Irregular','Annual','Quarterly','Monthly','Daily/real time'),
 'retention_clarity': ('None','Minimal','Basic guidelines','Moderate clarity','High clarity','Fully documented'),
 'regulation_coverage': ('None','Minimal','Partial','Moderate','Comprehensive but dated','Fully documented/current'),
 'consent_ethics': ('None','Minimal record','Partial','Moderate logs','Substantial','Fully documented'),
 'geo_restrictions': ('None','Basic mention','Partial','Moderate','Near-complete','Fully documented'),
 'sensitivity_classification': ('None','Basic flagging','Partial','Moderate','Near-complete','Fully classified'),
 'audits': ('None','Internal only','Basic certification','Occasional audit','Recent audit','Regular external audits'),
 'license_terms': ('None','Vague','Basic','Clear','Detailed','Industry-compliant'),
 'ethical_reviews': ('None','Informal','Partial','Moderate','Well-documented','Certified'),
 'redistribution': ('None','Unclear','Partial','Clear','Detailed','Fully documented'),
 'user_agreements': ('Non-compliant','Minimal','Partial','Mostly compliant','Fully compliant','Audited'),
}
assert set(RUBRICS)==set(DOC_FACTORS) and all(len(x)==6 for x in RUBRICS.values())

BANDS = {'high':(4,5),'medium':(3,4),'low':(1,2),'missing':(0,1)}
BAND_PROBABILITIES = (.52, .33, .10, .05)


def evaluate(scores):
    """Faithful reproduction of original non-ablation TADP 28-factor admission policy."""
    if set(scores) != ALL_FACTORS:
        raise ValueError('Missing/unexpected factor identifier')
    if any(type(v)!=int or not 0 <= v <= 5 for v in scores.values()):
        raise ValueError('Every factor must have an integer rubric level 0..5')
    d = {dim: sum(scores[f] for f in fs)/len(fs) for dim, fs in FACTORS.items()}
    hps = sum(WEIGHTS[dim]*d[dim] for dim in FACTORS)
    review_score = sum(WEIGHTS[dim]*d[dim] for dim in CRITICAL)/sum(WEIGHTS[dim] for dim in CRITICAL)
    if any(v==0 for v in scores.values()):
        accepted, path = False, 'BLOCKED_ZERO_FACTOR'
    elif any(x < MIN_DIM for x in d.values()):
        accepted, path = False, 'BLOCKED_DIMENSION_FLOOR'
    elif hps < REJECT_BELOW:
        accepted, path = False, 'AUTO_REJECT'
    elif hps >= DIRECT_ACCEPT_FROM:
        accepted, path = True, 'AUTO_ACCEPT'
    elif review_score >= REVIEW_MIDPOINT:
        accepted, path = True, 'REVIEW_ACCEPT'
    else:
        accepted, path = False, 'REVIEW_REJECT'
    return {'accepted':accepted, 'path':path, 'hps':round(hps,10),
            'review_score':round(review_score,10), 'dimensions':d}


def read_adult_archive():
    """Optionally reuse REAL computational D2 from user's archived TRAIN-only simulation.

    These are repeated assessments of 50 *simulated* clients, NOT 1,000 distinct
    verified providers. Reused only as plausible measured D2 backgrounds.
    """
    names = ('TADP_HPS28_Adult_Phased_Results_v5.zip',
             '/content/TADP_HPS28_Adult_Phased_Results_v5.zip',
             '/mnt/data/TADP_HPS28_Adult_Phased_Results_v5.zip')
    archive = next((Path(x) for x in names if Path(x).is_file()),None)
    if archive is None:
        return [], {'source':'synthetic_D2_fallback', 'archive_found':False,
                    'archived_reproduction_checked':False}
    pattern = re.compile(r'adult_alpha(0\.5|0\.1)/decisions_adult_seed(4[2-6])_(clean|degraded)_none\.json$')
    records=[]
    checks=0
    with zipfile.ZipFile(archive) as z:
        for name in sorted(z.namelist()):
            m=pattern.search(name)
            if not m: continue
            payload=json.loads(z.read(name))
            policy=payload.get('policy',{})
            for key, expected in (('min_dimension',MIN_DIM),('reject_below',REJECT_BELOW),
                                  ('direct_accept_from', DIRECT_ACCEPT_FROM)):
                if policy.get(key)!=expected: raise ValueError(f'Archive policy mismatch {name} {key}')
            if policy.get('critical_factor_floors'):
                raise ValueError('Cannot ignore archived critical factor floors')
            for cid, row in sorted(payload['decisions'].items()):
                d2={f:row['factor_scores'][f] for f in D2_FACTORS}
                base=evaluate({f:row['factor_scores'][f] for f in ALL_FACTORS})
                if (base['accepted']!=row['accepted'] or base['path']!=row['path'] or
                    abs(base['hps']-row['hps'])>1e-8):
                    raise AssertionError(f'Archived policy did not reproduce {name} {cid}')
                checks+=1
                records.append({'original_id':cid, 'alpha':m.group(1),
                                'condition':m.group(3), 'fl_seed':int(m.group(2)),
                                'd2':d2})
    if len(records)!=1000 or checks!=1000:
        raise ValueError(f'Expected 1,000 archived assessments; got {len(records)}')
    return records, {'source':'archived_Adult_TRAIN_D2', 'archive_found':True,
                     'archive_file':archive.name, 'archived_reproduction_checked':True,
                     'original_admission_matches':checks,
                     'source_caveat':'D2 reused from 50 simulated clients x 2 alpha x 2 conditions x 5 seeds; not 1000 independent TRAIN partitions.'}


def synthetic_d2(rng):
    """Explicit FALLBACK: synthetic *scores*, not real raw-TRAIN D2 calculations."""
    return {f: rng.choices(range(6), weights=(2,4,9,15,30,40), k=1)[0]
            if f in ('feature_distribution_consistency','feature_category_coverage')
            else rng.choices(range(6), weights=(1,1,2,4,20,72), k=1)[0]
            for f in D2_FACTORS}


def create_simulated_cases(archived):
    rng=random.Random(EVIDENCE_SEED)
    cases=[]
    for i in range(N_PROFILES):
        band=rng.choices(tuple(BANDS),weights=BAND_PROBABILITIES,k=1)[0]
        lo,hi=BANDS[band]
        # Each distinct case gets twenty independently selected *synthetic reviewer levels*
        # conditioned on the same documentary band, like the original benchmark.
        docs={f:rng.randint(lo,hi) for f in DOC_FACTORS}
        if archived:
            source=archived[i]  # 1,000 historical D2 assessment contexts, one per case
            d2=source['d2']
            origin={k:source[k] for k in ('original_id','alpha','condition','fl_seed')}
        else:
            d2=synthetic_d2(rng)
            origin={'original_id':'', 'alpha':'synthetic',
                    'condition':'synthetic', 'fl_seed':''}
        complete={**docs,**d2}
        baseline=evaluate(complete)
        cases.append({'id':f'SIM-{i+1:04d}','tier':band,'docs':docs,'d2':d2,
                      'origin':origin,'scores':complete,'baseline':baseline})
    return cases


def write_csv(path, rows, fieldnames):
    with Path(path).open('w',newline='',encoding='utf8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames,extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)


def pct(n,d):return 100*n/d if d else 0


def report_and_outputs(cases, source_info):
    RESULTS_DIR.mkdir(parents=True,exist_ok=True)
    library=[]
    baseline_rows=[]
    scenario_rows=[]
    factor_stats=defaultdict(lambda: Counter())
    dim_stats=defaultdict(lambda: Counter())
    error_type_stats=defaultdict(lambda:Counter())
    baseline_counts=Counter()
    case_flips=defaultdict(set)
    scenario_per_case=Counter()
    factor_effects={f:WEIGHTS[DIM_BY_FACTOR[f]]/len(FACTORS[DIM_BY_FACTOR[f]]) for f in DOC_FACTORS}

    for case in cases:
        b=case['baseline']
        baseline_counts[(case['tier'],'accept' if b['accepted'] else 'reject')]+=1
        library_row={'synthetic_assessment_id':case['id'], 'profile_band':case['tier'],
                     'authentic_uploaded_documents':False,
                     'reviewer':'SIMULATED_NOT_HUMAN',
                     'profile_seed':EVIDENCE_SEED}
        for f in DOC_FACTORS:
            library_row[f+'_level']=case['docs'][f]
            library_row[f+'_descriptor']=RUBRICS[f][case['docs'][f]]
        library.append(library_row)
        baseline_row={'assessment_id':case['id'],'band':case['tier'],
                      **case['origin'],
                      'baseline_accepted':int(b['accepted']),
                      'baseline_path':b['path'], 'baseline_hps':b['hps'],
                      'baseline_review_score':b['review_score']}
        baseline_row.update({d+'_mean':round(b['dimensions'][d],6) for d in FACTORS})
        baseline_row.update({'D2_'+f:case['d2'][f] for f in D2_FACTORS})
        baseline_rows.append(baseline_row)
        for f in DOC_FACTORS:
            old=case['docs'][f]
            for delta in (-1,1):
                new=old+delta
                if not 0<=new<=5:continue
                altered=case['scores'].copy()
                altered[f]=new
                v=evaluate(altered)
                flipped=(v['accepted']!=b['accepted'])
                direction=('reject_to_accept' if (flipped and v['accepted']) else
                           'accept_to_reject' if flipped else 'no_change')
                evidence_case=('unsupported_0_to_1' if old==0 and delta==1 else
                               'supported_1_to_0' if old==1 and delta==-1 else
                               'established_levels_only')
                row={'assessment_id':case['id'],'profile_band':case['tier'],
                     'factor':f,'dimension':DIM_BY_FACTOR[f],
                     'change':delta,'original_level':old,'perturbed_level':new,
                     'original_descriptor':RUBRICS[f][old],
                     'perturbed_descriptor':RUBRICS[f][new],
                     'evidence_boundary_type':evidence_case,
                     'baseline_accepted':int(b['accepted']),
                     'perturbed_accepted':int(v['accepted']),
                     'baseline_path':b['path'],'perturbed_path':v['path'],
                     'baseline_hps':b['hps'],'perturbed_hps':v['hps'],
                     'delta_hps':round(v['hps']-b['hps'],10),
                     'admission_changed':int(flipped),'transition':direction,
                     'original_D2_source':source_info['source']}
                scenario_rows.append(row)
                scenario_per_case[case['id']]+=1
                if flipped:case_flips[case['id']].add(f)
                for counts in (factor_stats[f],dim_stats[DIM_BY_FACTOR[f]],error_type_stats[evidence_case]):
                    counts['tested']+=1
                    counts['changed']+=int(flipped)
                    counts[direction]+=1
                    counts['baseline_accepted']+=int(b['accepted'])
                    counts['baseline_rejected']+=int(not b['accepted'])
                    counts['upward_tests']+=int(delta==1)
                    counts['downward_tests']+=int(delta==-1)
                    counts['upward_changes']+=int(flipped and delta==1)
                    counts['downward_changes']+=int(flipped and delta==-1)
                # If a single-factor mismatch crosses accept/reject, record the case at-risk.

    total=len(scenario_rows)
    n_flips=sum(r['admission_changed'] for r in scenario_rows)
    ra=sum(r['transition']=='reject_to_accept' for r in scenario_rows)
    ar=sum(r['transition']=='accept_to_reject' for r in scenario_rows)
    orig_accept=sum(c['baseline']['accepted'] for c in cases)

    # Factual mathematical property: a +/- 1 score always changes HPS by +/- weight / # factors.
    for f in DOC_FACTORS:
        related=[r for r in scenario_rows if r['factor']==f]
        if related and any(abs(abs(r['delta_hps'])-factor_effects[f])>1e-8 for r in related):
            raise AssertionError(f'Unexpected HPS score shift for {f}')

    factor_rows=[]
    for f in DOC_FACTORS:
        c=factor_stats[f]
        factor_rows.append({'factor':f,'dimension':DIM_BY_FACTOR[f],
                            'weight_per_level':round(factor_effects[f],9),
                            'tested_single_errors':c['tested'],
                            'admission_flips':c['changed'],
                            'flip_pct_of_feasible_shifts':round(pct(c['changed'],c['tested']),4),
                            'reject_to_accept':c['reject_to_accept'],
                            'accept_to_reject':c['accept_to_reject'],
                            'upward_shifts':c['upward_tests'],
                            'upward_flips':c['upward_changes'],
                            'downward_shifts':c['downward_tests'],
                            'downward_flips':c['downward_changes']})
    factor_rows.sort(key=lambda x:(-x['flip_pct_of_feasible_shifts'],-x['admission_flips'],x['factor']))

    dimension_rows=[]
    for d in ('D1','D3','D4','D5','D6'):
        c=dim_stats[d]
        dimension_rows.append({'dimension':d,'documentary_factors':len(FACTORS[d]),
                               'dimension_weight':WEIGHTS[d],
                               'tested_single_errors':c['tested'],
                               'admission_flips':c['changed'],
                               'flip_pct_of_feasible_shifts':round(pct(c['changed'],c['tested']),4),
                               'reject_to_accept':c['reject_to_accept'],
                               'accept_to_reject':c['accept_to_reject']})

    errors_by_type=[]
    for t,c in sorted(error_type_stats.items()):
        errors_by_type.append({'error_type':t,'tested_single_errors':c['tested'],
                                'admission_flips':c['changed'],
                                'flip_pct_of_feasible_shifts':round(pct(c['changed'],c['tested']),4),
                                'reject_to_accept':c['reject_to_accept'],
                                'accept_to_reject':c['accept_to_reject']})
    protected=[r for r in scenario_rows if r['evidence_boundary_type']=='established_levels_only']
    protected_flips=sum(r['admission_changed'] for r in protected)
    harmful_01=[r for r in scenario_rows if r['evidence_boundary_type']=='unsupported_0_to_1']

    # Distinguish "one error anywhere" vs the hypothetical "already-positive evidence only" scenario.
    near_threshold=[c for c in cases if (abs(c['baseline']['hps']-3.0)<=0.1 or
                                        abs(c['baseline']['hps']-3.5)<=0.1)]
    cohort_rows=[]
    for tier in BANDS:
        subset=[c for c in cases if c['tier']==tier]
        related=[r for r in scenario_rows if r['profile_band']==tier]
        cohort_rows.append({'profile_band':tier,'cases':len(subset),
                            'baseline_admitted':sum(c['baseline']['accepted'] for c in subset),
                            'shifts_tested':len(related),
                            'admission_flips':sum(r['admission_changed'] for r in related),
                            'flip_pct_of_shifts':round(pct(sum(r['admission_changed'] for r in related),len(related)),4)})
    num_vulnerable=len(case_flips)
    summary={
        'experiment':'Reviewer one-factor ±1-level rubric-error sensitivity',
        'policy_version':POLICY_VERSION,
        'experiment_type':'fully_synthetic_documentary_assessments',
        'simulated_independent_documentary_profiles':len(cases),
        'documentary_factors':len(DOC_FACTORS),
        'D2_factors_fixed_per_assessment':len(D2_FACTORS),
        'D2_source':source_info,
        'actual_human_reviewers':0,'authentic_documentary_evidence_packages':0,
        'repetitions_of_federated_training':0,
        'original_FL_experiment_count_unchanged':True,
        'seed_documentary':EVIDENCE_SEED,'seed_misc':ERROR_SEED,
        'tiers':{'high':.52,'medium':.33,'low':.10,'missing':.05},
        'baseline_admitted':orig_accept,
        'baseline_rejected':len(cases)-orig_accept,
        'baseline_admission_pct':round(pct(orig_accept,len(cases)),4),
        'feasible_single_factor_1_level_errors':total,
        'single_error_admission_flips':n_flips,
        'single_error_flip_pct_of_feasible_shifts':round(pct(n_flips,total),4),
        'single_error_reject_to_accept':ra,
        'single_error_accept_to_reject':ar,
        'single_error_reject_to_accept_pct_of_shifts':round(pct(ra,total),4),
        'single_error_accept_to_reject_pct_of_shifts':round(pct(ar,total),4),
        'cases_sensitive_to_at_least_one_single_factor_error':num_vulnerable,
        'pct_cases_with_any_single_factor_flip':round(pct(num_vulnerable,len(cases)),4),
        'already_supported_levels_shifts':len(protected),
        'already_supported_levels_admission_flips':protected_flips,
        'already_supported_levels_flip_pct':round(pct(protected_flips,len(protected)),4),
        'zero_to_one_false_support_shifts':len(harmful_01),
        'zero_to_one_false_support_admission_flips':sum(r['admission_changed'] for r in harmful_01),
        'cases_with_hps_within_0_1_of_any_threshold':len(near_threshold),
        'score_delta_per_factor':{f:round(factor_effects[f],9) for f in DOC_FACTORS},
        'unit_of_analysis_note':'The 1,000 synthetic documentary profiles are independent artificial cases; archived D2 contexts, when available, are reused from 50 virtual Adult clients and are not independent authentic providers.',
        'interpretation_limit':'Conditional decision robustness under a single simulated reviewer-level error; NOT measured human error rates, reviewer agreement, documentary authenticity, predictive performance, or proof of trustworthy input data.'
    }

    write_csv(RESULTS_DIR/'SIMULATED_Documentary_Library_1000.csv',library,list(library[0]))
    write_csv(RESULTS_DIR/'Baseline_1000.csv',baseline_rows,list(baseline_rows[0]))
    write_csv(RESULTS_DIR/'Exhaustive_One_Factor_PM1.csv',scenario_rows,list(scenario_rows[0]))
    write_csv(RESULTS_DIR/'Factor_Impact.csv',factor_rows,list(factor_rows[0]))
    write_csv(RESULTS_DIR/'Dimension_Impact.csv',dimension_rows,list(dimension_rows[0]))
    write_csv(RESULTS_DIR/'Evidence_Boundary_Impact.csv',errors_by_type,list(errors_by_type[0]))
    write_csv(RESULTS_DIR/'Profile_Band_Impact.csv',cohort_rows,list(cohort_rows[0]))
    (RESULTS_DIR/'Summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n',encoding='utf8')

    # Automatically produce two publication-oriented figures, separate axes and no subplots.
    try:
        import matplotlib.pyplot as plt
        sorted_chart=sorted(factor_rows,key=lambda r:r['flip_pct_of_feasible_shifts'])
        fig,ax=plt.subplots(figsize=(10,7.5))
        ax.barh([r['factor'] for r in sorted_chart],[r['flip_pct_of_feasible_shifts'] for r in sorted_chart])
        ax.set_xlabel('Admission changes / feasible single-factor ±1 changes (%)')
        ax.set_title('TADP documentary assessment uncertainty by factor (synthetic)')
        ax.grid(axis='x',alpha=.15)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR/'Figure_Factor_Admission_Impact.png',dpi=180)
        plt.close(fig)

        labels=['Unchanged','Rejected → admitted','Admitted → rejected']
        vals=[total-n_flips,ra,ar]
        fig,ax=plt.subplots(figsize=(8,4.5))
        ax.bar(labels,vals)
        ax.set_ylabel('Number of feasible single-factor ±1 scenarios')
        ax.set_title(f'TADP admission robustness across {total:,} simulated reviewer errors')
        ax.tick_params(axis='x',rotation=12)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR/'Figure_Admission_Transitions.png',dpi=180)
        plt.close(fig)
    except ImportError:
        print('Matplotlib not installed; CSV and JSON outputs still generated.')

    leaders=', '.join(f"{r['factor']} ({r['flip_pct_of_feasible_shifts']:.2f}%)" for r in factor_rows[:4])
    readme=f'''# TADP reviewer ±1 rubric-error sensitivity (AUTOMATIC SYNTHETIC SIMULATION)

## Main outcomes (ONE hypothetical factor-level error per assessment)

- Independently generated synthetic documentary profiles: **{len(cases):,}** (20 factors each).
- Real uploaded documents / actual authorized reviewers: **0 / 0**.
- D2 background: **{source_info['source']}**. D2 frozen for every perturbation.
- Baseline admitted: **{orig_accept:,} / {len(cases):,}** ({pct(orig_accept,len(cases)):.2f}%).
- Exhaustive feasible one-factor ±1 rubric-level changes: **{total:,}**.
- Admission flips: **{n_flips:,}** ({pct(n_flips,total):.3f}% of feasible shifts).
- Rejected→admitted: **{ra:,}**; admitted→rejected: **{ar:,}**.
- Synthetic cases where *at least one* documentary ±1 causes a flip: **{num_vulnerable:,} / {len(cases):,}** ({pct(num_vulnerable,len(cases)):.2f}%).
- Changes among *already-supported levels* (not 0→1 / 1→0): **{protected_flips:,} / {len(protected):,}** ({pct(protected_flips,len(protected)):.3f}%).
- False support from level 0→1: **{sum(r['admission_changed'] for r in harmful_01):,} / {len(harmful_01):,}** perturbations changed admission.
- Most decision-sensitive factors by equal-weighted feasible shift frequency: {leaders}.

## Design

1. Create 1,000 **synthetic reviewer-outcome profiles** with band probabilities 52% high (4–5), 33% medium (3–4), 10% low (1–2), and 5% missing (0–1). These levels are mapped to the **exact Appendix A descriptors**; this is not a library of authentic documentary files.
2. D2: when available, reuse eight original Adult TRAIN-only D2 scores from each of the 1,000 archived client-condition-seed assessments. The archive is itself 50 virtual clients x 2 alphas x 2 clean/degraded x 5 seeds; thus its D2 backgrounds are **not 1,000 independent providers**. Otherwise use clearly marked synthetic D2 score distributions.
3. Apply original TADP-HPS28/1.0 admission: zeros fail, dimension floor 2.5, HPS reject<3, direct accept>=3.5, intermediate critical weighted review >=3.25; no extra critical factor floors or disqualifiers.
4. For every documentary factor of every case, make one **single** valid -1 and/or +1 error. Only the selected factor changes, remaining 27 are fixed. Recalculate admission. Changes at 0→1 and 1→0 are shown separately.
5. Enumerate **all feasible changes** (not a Monte Carlo estimate of mistake rates). Counts are hypothetical scenario frequencies. No human review error probability is assumed; probabilities are conditional on uniform weighting of the listed feasible shifts, not calibrated to human behavior.

## Interpreting the outcome responsibly

A low flip percentage means the chosen deterministic admission policy resists one-step perturbations *for the simulated baseline distribution*. It DOES NOT show that reviewers agree, evidence is genuine, inputs comply with laws, or the governance scheme is universally robust. A case can flip at the hard 0-factor rule even if HPS changes only slightly. Human errors may be correlated, larger than one step, or biased toward certain factors. Values are descriptive and cannot be generalized to real institutions.

## Outputs

- `SIMULATED_Documentary_Library_1000.csv`: 20 Appendix A descriptors and levels for each synthetic case, explicit NOT HUMAN flag.
- `Baseline_1000.csv`: baseline HPS and admissions, frozen D2.
- `Exhaustive_One_Factor_PM1.csv`: every feasible one-factor +/-1 alternative and admission transition.
- `Factor_Impact.csv`: per documentary factor sensitivity and reject/accept direction.
- `Dimension_Impact.csv`: aggregated dimension sensitivity.
- `Evidence_Boundary_Impact.csv`: isolate level0→1, level1→0 vs already-positive transitions.
- `Profile_Band_Impact.csv`: group-level stratification.
- `Summary.json`: quantitative summary and reproducibility settings.
- `Figure_Factor_Admission_Impact.png`, `Figure_Admission_Transitions.png`: graphics if matplotlib is installed.

## Reproducibility

Run: `%run TADP_Reviewer_PM1_Sensitivity_AUTO.py` in Google Colab. **No input or menu.** ZIP auto-detected if present. Documentary RNG seed = {EVIDENCE_SEED}; simulated profiles = {N_PROFILES}. No FL model retraining, and the existing 590 original training evaluations are untouched.
'''
    (RESULTS_DIR/'REPORT.md').write_text(readme,encoding='utf8')

    with zipfile.ZipFile(RESULTS_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(RESULTS_DIR.iterdir()):
            if p.is_file():z.write(p,arcname=f'{RESULTS_DIR.name}/{p.name}')
    print('\\n'+'='*78)
    print('AUTOMATIC TADP DOCUMENTARY HUMAN-REVIEWER +/-1 ERROR SENSITIVITY')
    print('='*78)
    print('Documentary cases:           ',f'{len(cases):,}', '(SYNTHETIC; 0 actual documents / reviewers)')
    print('D2 source:                   ',source_info['source'])
    print('Original archive verified:   ',source_info.get('original_admission_matches',0),'of 1,000')
    print('Baseline admitted:           ',f'{orig_accept:,} / {len(cases):,} ({pct(orig_accept,len(cases)):.2f}%)')
    print('All feasible +/-1 scenarios: ',f'{total:,}')
    print('Admission decisions changed: ',f'{n_flips:,} ({pct(n_flips,total):.3f}%)')
    print('  rejected -> admitted:      ',f'{ra:,}')
    print('  admitted -> rejected:      ',f'{ar:,}')
    print('Cases sensitive to >=1 error:',f'{num_vulnerable:,} / {len(cases):,} ({pct(num_vulnerable,len(cases)):.2f}%)')
    print('No-missing-evidence category:',f'{protected_flips:,} / {len(protected):,} changed ({pct(protected_flips,len(protected)):.3f}%)')
    print('Most sensitive factors:      ',leaders)
    print('RESULTS ZIP:                 ',RESULTS_ZIP.resolve())
    print('REPORT:                      ',(RESULTS_DIR/'REPORT.md').resolve())
    print('NOTE: Scenario frequencies are NOT estimates of human reviewer error rates.')
    print('='*78)
    return summary


def main():
    archive,source_info=read_adult_archive()
    cases=create_simulated_cases(archive)
    return report_and_outputs(cases,source_info)


if __name__=='__main__':
    main()


\n==============================================================================
AUTOMATIC TADP DOCUMENTARY HUMAN-REVIEWER +/-1 ERROR SENSITIVITY
Documentary cases:            1,000 (SYNTHETIC; 0 actual documents / reviewers)
D2 source:                    archived_Adult_TRAIN_D2
Original archive verified:    1000 of 1,000
Baseline admitted:            462 / 1,000 (46.20%)
All feasible +/-1 scenarios:  34,400
Admission decisions changed:  53 (0.154%)
  rejected -> admitted:       13
  admitted -> rejected:       40
Cases sensitive to >=1 error: 5 / 1,000 (0.50%)
No-missing-evidence category: 53 / 32,377 changed (0.164%)
Most sensitive factors:       consent_ethics (0.29%), data_collection_lineage (0.29%), geo_restrictions (0.29%), source_reputation (0.29%)
RESULTS ZIP:                  /content/TADP_Reviewer_PM1_Sensitivity_RESULTS.zip
REPORT:                       /content/TADP_Reviewer_PM1_Results/REPORT.md
NOTE: Scenario frequencies are NOT estimates of human reviewer error rates.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""TADP-HPS28: AUTOMATIC simulated HUMAN reviewer +/-1 rubric uncertainty.

ONE-RUN, ZERO PROMPTS, NO AI / API, NO HUMAN INPUT.

Creates 1,000 independent synthetic documentary assessment profiles (20 Appendix A
rubric descriptors per profile). For D2 it preferentially REPLAYS the archived
TRAIN-only eight-factor scores from the user's original Adult benchmark (1,000
client-condition-seed assessments = 50 virtual clients x 2 alphas x 2 data
conditions x 5 seeds). If the ZIP cannot be found, a clearly marked synthetic
D2 *score* fallback is used. No raw data, real evidence documents, or human
reviewers are processed or impersonated.

For each synthetic assessment, exhaustively enumerates ALL feasible changes to
ONE, TWO, or THREE DIFFERENT documentary factor levels by exactly +/-1, keeping
the eight D2 factors unchanged. Additionally creates 400 deliberately boundary-
enriched synthetic examples near HPS, review, dimension and zero-factor gates.
This is a conditional deterministic stress test, not an estimate of mistakes.

All outputs are descriptive and conditional on a hypothetical single-factor
review error. A flip rate is NOT an empirical probability of human error.

RUN IN COLAB: upload file then run:
    %run TADP_Reviewer_PM1_Sensitivity_AUTO.py
or in terminal: python TADP_Reviewer_PM1_Sensitivity_AUTO.py
No arguments, input prompts or API credentials are needed.

Dependencies: Python 3.10+, numpy, matplotlib (installed in Colab).
"""
from __future__ import annotations

import csv
import json
import math
import random
import re
import statistics
import sys
import zipfile
import numpy as np
from collections import Counter, defaultdict
from itertools import combinations, product
from datetime import datetime, timezone
from pathlib import Path

N_PROFILES = 1000
EVIDENCE_SEED = 7702
ERROR_SEED = 20261108
N_BOUNDARY_PER_TYPE = 80
NEAR_HPS_WINDOW = 0.10
NEAR_REVIEW_WINDOW = 0.10
NEAR_DIM_WINDOW = 0.10
RESULTS_DIR = Path('TADP_Reviewer_Multi_Error_Results')
RESULTS_ZIP = Path('TADP_Reviewer_Multi_Error_Results.zip')

POLICY_VERSION = 'TADP-HPS-28/1.0'
MIN_DIM = 2.5
REJECT_BELOW = 3.0
DIRECT_ACCEPT_FROM = 3.5
REVIEW_MIDPOINT = (REJECT_BELOW + DIRECT_ACCEPT_FROM) / 2

FACTORS = {
 'D1': ('source_reputation', 'data_controller', 'data_objective', 'data_collection_lineage'),
 'D2': ('completeness', 'duplication_rate', 'value_validity_error_rate', 'type_consistency',
        'label_integrity', 'feature_distribution_consistency', 'feature_category_coverage',
        'structural_constraint_integrity'),
 'D3': ('data_dictionary', 'version_logs', 'collection_protocol', 'definition_updates'),
 'D4': ('data_freshness', 'scheduled_refresh', 'retention_clarity'),
 'D5': ('regulation_coverage', 'consent_ethics', 'geo_restrictions', 'sensitivity_classification', 'audits'),
 'D6': ('license_terms', 'ethical_reviews', 'redistribution', 'user_agreements'),
}
WEIGHTS = {'D1': 0.25, 'D2': 0.15, 'D3': 0.10, 'D4': 0.10, 'D5': 0.30, 'D6': 0.10}
CRITICAL = ('D1', 'D2', 'D5', 'D6')
DOC_FACTORS = sorted(f for d, factors in FACTORS.items() if d != 'D2' for f in factors)
D2_FACTORS = FACTORS['D2']
ALL_FACTORS = set(DOC_FACTORS) | set(D2_FACTORS)
DIM_BY_FACTOR = {f: d for d, fs in FACTORS.items() for f in fs}
assert len(DOC_FACTORS) == 20 and len(ALL_FACTORS) == 28 and abs(sum(WEIGHTS.values())-1)<1e-12

# Exact user-supplied Appendix A documentary descriptors; order = deterministic level 0..5.
RUBRICS = {
 'source_reputation': ('No information','Poor','Limited evidence','Average, partially trusted','Well-documented, reliable','Highly reputable, verified'),
 'data_controller': ('No documented controller','Unclear','Partially clear','Moderately clear','Mostly clear','Fully documented'),
 'data_objective': ('None','Vague','Partial','General but unclear','Mostly explicit','Fully explicit and justified'),
 'data_collection_lineage': ('Origin unknown','Informal/unverifiable origin','Partially documented path','Documented with limited traceability','Well-documented traceable path','Fully source-linked/versioned/auditable'),
 'data_dictionary': ('None','Minimal outline','Partial coverage','Moderate coverage','Near-complete','Fully detailed'),
 'version_logs': ('None','Minimal logs','Occasional logs','Regular logs','Near-complete','Full version history'),
 'collection_protocol': ('None','Vague','Partial','General methods','Well-defined','Fully transparent'),
 'definition_updates': ('None','Rare','Occasional','Regular but basic','Frequent','Real-time documented'),
 'data_freshness': ('>5 years','2-5 years','1-2 years','6-12 months','1-6 months','Current'),
 'scheduled_refresh': ('Never','Irregular','Annual','Quarterly','Monthly','Daily/real time'),
 'retention_clarity': ('None','Minimal','Basic guidelines','Moderate clarity','High clarity','Fully documented'),
 'regulation_coverage': ('None','Minimal','Partial','Moderate','Comprehensive but dated','Fully documented/current'),
 'consent_ethics': ('None','Minimal record','Partial','Moderate logs','Substantial','Fully documented'),
 'geo_restrictions': ('None','Basic mention','Partial','Moderate','Near-complete','Fully documented'),
 'sensitivity_classification': ('None','Basic flagging','Partial','Moderate','Near-complete','Fully classified'),
 'audits': ('None','Internal only','Basic certification','Occasional audit','Recent audit','Regular external audits'),
 'license_terms': ('None','Vague','Basic','Clear','Detailed','Industry-compliant'),
 'ethical_reviews': ('None','Informal','Partial','Moderate','Well-documented','Certified'),
 'redistribution': ('None','Unclear','Partial','Clear','Detailed','Fully documented'),
 'user_agreements': ('Non-compliant','Minimal','Partial','Mostly compliant','Fully compliant','Audited'),
}
assert set(RUBRICS)==set(DOC_FACTORS) and all(len(x)==6 for x in RUBRICS.values())

BANDS = {'high':(4,5),'medium':(3,4),'low':(1,2),'missing':(0,1)}
BAND_PROBABILITIES = (.52, .33, .10, .05)


def evaluate(scores):
    """Faithful reproduction of original non-ablation TADP 28-factor admission policy."""
    if set(scores) != ALL_FACTORS:
        raise ValueError('Missing/unexpected factor identifier')
    if any(type(v)!=int or not 0 <= v <= 5 for v in scores.values()):
        raise ValueError('Every factor must have an integer rubric level 0..5')
    d = {dim: sum(scores[f] for f in fs)/len(fs) for dim, fs in FACTORS.items()}
    hps = sum(WEIGHTS[dim]*d[dim] for dim in FACTORS)
    review_score = sum(WEIGHTS[dim]*d[dim] for dim in CRITICAL)/sum(WEIGHTS[dim] for dim in CRITICAL)
    if any(v==0 for v in scores.values()):
        accepted, path = False, 'BLOCKED_ZERO_FACTOR'
    elif any(x < MIN_DIM - 1e-10 for x in d.values()):
        accepted, path = False, 'BLOCKED_DIMENSION_FLOOR'
    elif hps < REJECT_BELOW - 1e-10:
        accepted, path = False, 'AUTO_REJECT'
    elif hps >= DIRECT_ACCEPT_FROM - 1e-10:
        accepted, path = True, 'AUTO_ACCEPT'
    elif review_score >= REVIEW_MIDPOINT - 1e-10:
        accepted, path = True, 'REVIEW_ACCEPT'
    else:
        accepted, path = False, 'REVIEW_REJECT'
    return {'accepted':accepted, 'path':path, 'hps':round(hps,10),
            'review_score':round(review_score,10), 'dimensions':d}


def read_adult_archive():
    """Optionally reuse REAL computational D2 from user's archived TRAIN-only simulation.

    These are repeated assessments of 50 *simulated* clients, NOT 1,000 distinct
    verified providers. Reused only as plausible measured D2 backgrounds.
    """
    names = ('TADP_HPS28_Adult_Phased_Results_v5.zip',
             '/content/TADP_HPS28_Adult_Phased_Results_v5.zip',
             '/mnt/data/TADP_HPS28_Adult_Phased_Results_v5.zip',
             str(Path(__file__).resolve().parent / 'TADP_HPS28_Adult_Phased_Results_v5.zip') if '__file__' in globals() else '')
    archive = next((Path(x) for x in names if Path(x).is_file()),None)
    if archive is None:
        return [], {'source':'synthetic_D2_fallback', 'archive_found':False,
                    'archived_reproduction_checked':False}
    pattern = re.compile(r'adult_alpha(0\.5|0\.1)/decisions_adult_seed(4[2-6])_(clean|degraded)_none\.json$')
    records=[]
    checks=0
    with zipfile.ZipFile(archive) as z:
        for name in sorted(z.namelist()):
            m=pattern.search(name)
            if not m: continue
            payload=json.loads(z.read(name))
            policy=payload.get('policy',{})
            for key, expected in (('min_dimension',MIN_DIM),('reject_below',REJECT_BELOW),
                                  ('direct_accept_from', DIRECT_ACCEPT_FROM)):
                if policy.get(key)!=expected: raise ValueError(f'Archive policy mismatch {name} {key}')
            if policy.get('critical_factor_floors'):
                raise ValueError('Cannot ignore archived critical factor floors')
            for cid, row in sorted(payload['decisions'].items()):
                d2={f:row['factor_scores'][f] for f in D2_FACTORS}
                base=evaluate({f:row['factor_scores'][f] for f in ALL_FACTORS})
                if (base['accepted']!=row['accepted'] or base['path']!=row['path'] or
                    abs(base['hps']-row['hps'])>1e-8):
                    raise AssertionError(f'Archived policy did not reproduce {name} {cid}')
                checks+=1
                records.append({'original_id':cid, 'alpha':m.group(1),
                                'condition':m.group(3), 'fl_seed':int(m.group(2)),
                                'd2':d2})
    if len(records)!=1000 or checks!=1000:
        raise ValueError(f'Expected 1,000 archived assessments; got {len(records)}')
    return records, {'source':'archived_Adult_TRAIN_D2', 'archive_found':True,
                     'archive_file':archive.name, 'archived_reproduction_checked':True,
                     'original_admission_matches':checks,
                     'source_caveat':'D2 reused from 50 simulated clients x 2 alpha x 2 conditions x 5 seeds; not 1000 independent TRAIN partitions.'}


def synthetic_d2(rng):
    """Explicit FALLBACK: synthetic *scores*, not real raw-TRAIN D2 calculations."""
    return {f: rng.choices(range(6), weights=(2,4,9,15,30,40), k=1)[0]
            if f in ('feature_distribution_consistency','feature_category_coverage')
            else rng.choices(range(6), weights=(1,1,2,4,20,72), k=1)[0]
            for f in D2_FACTORS}


def create_simulated_cases(archived):
    rng=random.Random(EVIDENCE_SEED)
    cases=[]
    for i in range(N_PROFILES):
        band=rng.choices(tuple(BANDS),weights=BAND_PROBABILITIES,k=1)[0]
        lo,hi=BANDS[band]
        # Each distinct case gets twenty independently selected *synthetic reviewer levels*
        # conditioned on the same documentary band, like the original benchmark.
        docs={f:rng.randint(lo,hi) for f in DOC_FACTORS}
        if archived:
            source=archived[i]  # 1,000 historical D2 assessment contexts, one per case
            d2=source['d2']
            origin={k:source[k] for k in ('original_id','alpha','condition','fl_seed')}
        else:
            d2=synthetic_d2(rng)
            origin={'original_id':'', 'alpha':'synthetic',
                    'condition':'synthetic', 'fl_seed':''}
        complete={**docs,**d2}
        baseline=evaluate(complete)
        cases.append({'id':f'SIM-{i+1:04d}','tier':band,'docs':docs,'d2':d2,
                      'origin':origin,'scores':complete,'baseline':baseline})
    return cases




def write_csv(path, rows, fieldnames=None):
    path = Path(path)
    if not rows:
        return
    if fieldnames is None:
        fieldnames = list(rows[0].keys())
    with path.open('w', newline='', encoding='utf-8') as handle:
        out = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction='ignore')
        out.writeheader()
        out.writerows(rows)


def rate(n, denominator):
    return round((100 * n / denominator), 6) if denominator else 0.0


# All +1/-1 alternatives for k different documentary factors, each used once.
# 760 2-factor combos and 9120 3-factor combos BEFORE boundary filtering.
DOC_NAMES = tuple(DOC_FACTORS)
DIM_NAMES = tuple(FACTORS)
DIMS = np.array([DIM_NAMES.index(DIM_BY_FACTOR[f]) for f in DOC_NAMES], dtype=np.int8)
LEVEL_HPS = np.array([WEIGHTS[DIM_BY_FACTOR[f]] / len(FACTORS[DIM_BY_FACTOR[f]])
                      for f in DOC_NAMES], dtype=float)
CRIT_DENOM = sum(WEIGHTS[d] for d in CRITICAL)
LEVEL_REVIEW = np.array([LEVEL_HPS[i] / CRIT_DENOM if DIM_BY_FACTOR[f] in CRITICAL else 0.0
                         for i, f in enumerate(DOC_NAMES)], dtype=float)
DIM_SIZES = np.array([len(FACTORS[d]) for d in DIM_NAMES], dtype=float)


def all_combinations():
    variants = {}
    for k in (1, 2, 3):
        indexes = []
        shifts = []
        for idx in combinations(range(20), k):
            for sgn in product((-1, 1), repeat=k):
                indexes.append(idx)
                shifts.append(sgn)
        variants[k] = (np.asarray(indexes, dtype=np.int8),
                       np.asarray(shifts, dtype=np.int8))
    assert len(variants[1][0]) == 40
    assert len(variants[2][0]) == 760
    assert len(variants[3][0]) == 9120
    return variants


def evaluate_exhaustively(case, combos, k):
    """Fast exact enumeration using analytic score deltas and array operations."""
    indexes, changes = combos
    levels = np.asarray([case['docs'][f] for f in DOC_NAMES], dtype=np.int8)
    old = levels[indexes]
    feasible = np.all((old + changes >= 0) & (old + changes <= 5), axis=1)
    idx = indexes[feasible]
    delta = changes[feasible]
    old = old[feasible]
    n = len(idx)
    if not n:
        return {'n':0, 'flips':0, 'r2a':0, 'a2r':0, 'changed_idx':np.array([])}

    base = case['baseline']
    base_dims = np.asarray([base['dimensions'][d] for d in DIM_NAMES], dtype=float)
    base_hps = sum(WEIGHTS[d]*base['dimensions'][d] for d in DIM_NAMES)
    base_review = sum(WEIGHTS[d]*base['dimensions'][d] for d in CRITICAL)/CRIT_DENOM
    new_hps = base_hps + np.sum(LEVEL_HPS[idx] * delta, axis=1)
    new_review = base_review + np.sum(LEVEL_REVIEW[idx] * delta, axis=1)

    # Score 0 anywhere is a hard rejection. D2 remains fixed, including zeros.
    n_original_zeros = sum(s == 0 for s in case['scores'].values())
    removed_zeros = np.sum((old == 0) & (delta == 1), axis=1)
    created_zeros = np.sum((old == 1) & (delta == -1), axis=1)
    no_zeros = (n_original_zeros - removed_zeros + created_zeros) == 0

    new_dims = np.broadcast_to(base_dims, (n, len(DIM_NAMES))).copy()
    for d_id in range(len(DIM_NAMES)):
        new_dims[:,d_id] += np.sum((DIMS[idx] == d_id)*delta, axis=1) / DIM_SIZES[d_id]
    floor_ok = np.all(new_dims >= MIN_DIM - 1e-12, axis=1)
    # Rounding to 10 decimals avoids machine-epsilon ambiguity exactly at a threshold.
    # Same 1e-10 threshold tolerance is used in evaluate(). Archive decisions are rechecked.
    new_hps_check = np.round(new_hps, 10)
    new_review_check = np.round(new_review, 10)
    accepted = no_zeros & floor_ok & (
        (new_hps_check >= DIRECT_ACCEPT_FROM) |
        ((new_hps_check >= REJECT_BELOW) & (new_review_check >= REVIEW_MIDPOINT))
    )
    flip = accepted != base['accepted']
    r2a = flip & accepted
    a2r = flip & ~accepted
    changed = np.flatnonzero(flip)

    return {
        'n':int(n), 'flips':int(flip.sum()), 'r2a':int(r2a.sum()),
        'a2r':int(a2r.sum()), 'changed_idx':changed,
        'idx':idx, 'deltas':delta, 'new_hps':new_hps, 'new_review':new_review,
        'new_accepted':accepted, 'flip':flip,
        'any_level_0_to_1':np.any((old==0)&(delta==1),axis=1),
        'any_level_1_to_0':np.any((old==1)&(delta==-1),axis=1),
    }


def initial_doc_levels(level, rng):
    return {f:level for f in DOC_NAMES}


def candidate_for_gate(d2, target, rng):
    """Synthetic near-gate cases (NOT real reviews). D2 scores are frozen."""
    label = target
    if target == 'zero_gate':
        docs = initial_doc_levels(4, rng)
        f = DOC_NAMES[rng.randrange(len(DOC_NAMES))]
        docs[f] = 0
        return docs
    if target == 'dimension_2_5':
        docs = initial_doc_levels(4, rng)
        d = rng.choice(('D1', 'D3', 'D6'))
        fs = [f for f in DOC_NAMES if DIM_BY_FACTOR[f] == d]
        rng.shuffle(fs)
        for j, f in enumerate(fs):
            docs[f] = 2 if j < 2 else 3
        return docs

    docs = initial_doc_levels(3 if target in ('hps_3_0','review_3_25') else 4, rng)
    if target == 'hps_3_0': target_val = 3.0
    elif target == 'hps_3_5': target_val = 3.5
    elif target == 'review_3_25': target_val = 3.25
    else: raise ValueError(target)

    def loss(ds):
        result=evaluate({**ds,**d2})
        # Never artificially create level zero or dimension-floor violations here.
        floor_penalty=sum(max(0, MIN_DIM-x)*100 for x in result['dimensions'].values())
        if target == 'review_3_25':
            # Also keep HPS inside the automated-review decision interval.
            hps_out=max(0, REJECT_BELOW-result['hps'])+max(0,result['hps']-DIRECT_ACCEPT_FROM)
            return abs(result['review_score']-target_val)+50*hps_out+floor_penalty
        return abs(result['hps']-target_val)+floor_penalty

    # Discrete coordinate search. Inputs and operation ordering deterministic.
    best_loss=loss(docs)
    order=list(DOC_NAMES)
    for _ in range(10):
        rng.shuffle(order)
        improved=False
        for f in order:
            score=docs[f]
            choices=[]
            for movement in (-1,1):
                new=score+movement
                if 1<=new<=5:
                    docs[f]=new
                    objective=loss(docs)
                    choices.append((objective,new))
            docs[f]=score
            if choices:
                objective,new=min(choices)
                if objective < best_loss-1e-10:
                    docs[f]=new
                    best_loss=objective
                    improved=True
        if not improved:
            break
    return docs


def build_boundary_cases(archived):
    rng=random.Random(ERROR_SEED+111)
    candidates=[r for r in archived if all(v>0 for v in r['d2'].values())] if archived else []
    targets=('hps_3_0','hps_3_5','review_3_25','dimension_2_5','zero_gate')
    cases=[]
    for goal in targets:
        for i in range(N_BOUNDARY_PER_TYPE):
            origin={}
            if candidates:
                record=candidates[rng.randrange(len(candidates))]
                d2=record['d2'].copy()
                origin={x:record[x] for x in ('original_id','alpha','condition','fl_seed')}
            else:
                # Synthetic fallback keeps all eight D2 metrics positive by design.
                d2={f:rng.choice((3,4,4,5,5)) for f in D2_FACTORS}
                origin={'original_id':'','alpha':'synthetic','condition':'synthetic','fl_seed':''}
            docs=candidate_for_gate(d2,goal,rng)
            combined={**docs,**d2}
            base=evaluate(combined)
            cases.append({'id':f'BOUNDARY-{goal}-{i+1:03d}', 'group':'targeted_boundary',
                          'goal':goal,'tier':'purpose_built', 'docs':docs, 'd2':d2,
                          'scores':combined,'origin':origin,'baseline':base})
    return cases


def boundary_properties(case):
    b=case['baseline']; dims=b['dimensions']
    d2_zero=any(v==0 for v in case['d2'].values())
    doc_zero=any(v==0 for v in case['docs'].values())
    hps_near=abs(b['hps']-3.0)<=NEAR_HPS_WINDOW or abs(b['hps']-3.5)<=NEAR_HPS_WINDOW
    review_near=abs(b['review_score']-REVIEW_MIDPOINT)<=NEAR_REVIEW_WINDOW
    floor_near=any(abs(v-MIN_DIM)<=NEAR_DIM_WINDOW+1e-9 for v in dims.values())
    return {'near_hps':hps_near,'near_review':review_near,'near_floor':floor_near,
            'zero_gate':doc_zero or d2_zero,
            'D2_zero':d2_zero,'documentary_zero':doc_zero,
            'no_zero_factors':not(doc_zero or d2_zero),
            'near_any_gate':hps_near or review_near or floor_near or doc_zero or d2_zero}


def one_case_row(case, props):
    b=case['baseline']
    return {'case_id':case['id'],'population':case['group'],'target':case['goal'],
            'tier':case['tier'],**case['origin'],
            'HPS':b['hps'],'review_score':b['review_score'],
            'admitted':int(b['accepted']),'decision_path':b['path'],
            **{f'DIM_{d}':round(b['dimensions'][d],8) for d in DIM_NAMES},
            **{k:int(v) for k,v in props.items()}}


def summarize_scenario(row, out, k):
    return {'population':row['population'], 'target':row['target'], 'error_count':k,
            'scenarios':out['n'], 'admission_changes':out['flips'],
            'flip_pct':rate(out['flips'],out['n']),
            'rejected_to_admitted':out['r2a'], 'admitted_to_rejected':out['a2r']}


def verify_vectorized_policy(cases, combo_dict):
    """Audit a reproducible sample of vectorized combinations with scalar TADP."""
    rng=random.Random(20261109)
    sampled=[c for c in cases if c['group']=='main_1000' and c['baseline']['accepted']][:20]
    sampled += [c for c in cases if c['group']=='main_1000' and not c['baseline']['accepted']][:20]
    for goal in ('hps_3_0','hps_3_5','review_3_25','dimension_2_5','zero_gate'):
        sampled += [c for c in cases if c['goal']==goal][:20]
    examined=0
    for case in sampled:
        for k in (1,2,3):
            out=evaluate_exhaustively(case,combo_dict[k],k)
            selected=set(rng.sample(range(out['n']),min(15,out['n'])))
            selected.update(int(j) for j in out['changed_idx'][:15])
            for j in selected:
                altered=case['scores'].copy()
                for i,d in zip(out['idx'][j],out['deltas'][j]):
                    altered[DOC_NAMES[int(i)]]+=int(d)
                expected=evaluate(altered)
                if expected['accepted']!=bool(out['new_accepted'][j]):
                    raise AssertionError(f'Vectorized/scalar mismatch: {case["id"]}, k={k}, row={j}')
                examined+=1
    return examined


def run_all():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    archived, source = read_adult_archive()
    main_cases=create_simulated_cases(archived)
    for c in main_cases:
        c['group']='main_1000';c['goal']='original_documentary_distribution'
    stress_cases=build_boundary_cases(archived)
    all_cases=main_cases+stress_cases
    combos=all_combinations()
    scalar_validation_checks=verify_vectorized_policy(all_cases,combos)
    rows=[]; per_case=[]; sample_flips=[]
    agg=defaultdict(Counter)
    props_per_case={}
    factor_stats=defaultdict(Counter)
    evidence_gate_stats=defaultdict(Counter)

    library=[];baseline_rows=[]
    for case in all_cases:
        props=boundary_properties(case)
        props_per_case[case['id']]=props
        baseline_rows.append(one_case_row(case,props))
        library_entry={'case_id':case['id'],'population':case['group'],
                       'target':case['goal'],'documentary_source':'SYNTHETIC_ONLY',
                       'authentic_uploaded_documents':False,'human_reviewers':0}
        library_entry.update({f+'_level':case['docs'][f] for f in DOC_NAMES})
        library_entry.update({f+'_descriptor':RUBRICS[f][case['docs'][f]] for f in DOC_NAMES})
        library.append(library_entry)

        for k in (1,2,3):
            out=evaluate_exhaustively(case, combos[k], k)
            row=summarize_scenario({'population':case['group'],'target':case['goal']},out,k)
            per_case.append({'case_id':case['id'],**row,
                             'any_flip':int(out['flips']>0),**{x:int(v) for x,v in props.items()}})
            zero_up=out['any_level_0_to_1']
            zero_down=out['any_level_1_to_0']
            for label, flag in [('0_to_1_only',zero_up & ~zero_down),
                                ('1_to_0_only',zero_down & ~zero_up),
                                ('both_zero_boundary_directions',zero_up & zero_down),
                                ('no_zero_boundary_crossing',~zero_up & ~zero_down)]:
                st=evidence_gate_stats[(case['group'],k,label)]
                st['scenarios']+=int(flag.sum())
                st['flips']+=int(np.sum(out['flip'] & flag))
            group_keys=[(case['group'],'ALL'),(case['group'],case['goal'])]
            # Predefined overlapping, carefully labeled conditional subgroups.
            if case['group']=='main_1000':
                for flag in ('D2_zero','documentary_zero','no_zero_factors','near_hps',
                             'near_review','near_floor','near_any_gate'):
                    if props[flag]: group_keys.append((case['group'],flag))
            for groupkey in group_keys:
                a=agg[(groupkey[0],groupkey[1],k)]
                a['case_count']+=1
                a['scenario_count']+=out['n'];a['flips']+=out['flips']
                a['r2a']+=out['r2a'];a['a2r']+=out['a2r']
                a['vulnerable_cases']+=int(out['flips']>0)

            # Per-factor report for single-error perturbations only: equal exposure.
            if k==1 and case['group']=='main_1000':
                idx, ds=out['idx'],out['deltas']
                for j, f in enumerate(DOC_NAMES):
                    selected=(idx[:,0]==j)
                    c=factor_stats[f]
                    c['scenarios']+=int(selected.sum())
                    c['flips']+=int(np.sum(out['flip'] & selected))

            # Keep a small reproducible set of concrete examples. No 10M-row CSV.
            if len(sample_flips)<250 and out['flips']:
                for j in out['changed_idx'][:max(0,250-len(sample_flips))]:
                    factors=[]
                    for f_idx, delta in zip(out['idx'][j],out['deltas'][j]):
                        name=DOC_NAMES[int(f_idx)]
                        factors.append(f'{name}:{case["docs"][name]}->{case["docs"][name]+int(delta)}')
                    sample_flips.append({'case_id':case['id'],'population':case['group'],
                                         'target':case['goal'],'error_count':k,
                                         'changes':'; '.join(factors),
                                         'original_hps':case['baseline']['hps'],
                                         'perturbed_hps':round(float(out['new_hps'][j]),8),
                                         'original_accepted':int(case['baseline']['accepted']),
                                         'perturbed_accepted':int(out['new_accepted'][j])})

    # Baseline consistency: original user experiment's headline values are expected
    # with the same archive, code, and documentary seed.
    if archived:
        assert sum(c['baseline']['accepted'] for c in main_cases)==462, 'Original 1000-case baseline does not match'
        a=agg[('main_1000','ALL',1)]
        assert a['scenario_count']==34400 and a['flips']==53 and a['vulnerable_cases']==5, (
            'Original published single-factor results did not reproduce',dict(a))

    summary=[]
    for (pop,group,k),a in sorted(agg.items()):
        summary.append({'population':pop,'subgroup':group,'error_count':k,
                        'cases':a['case_count'],'feasible_scenarios':a['scenario_count'],
                        'admission_changes':a['flips'],
                        'flip_pct_of_scenarios':rate(a['flips'],a['scenario_count']),
                        'rejected_to_admitted':a['r2a'],
                        'admitted_to_rejected':a['a2r'],
                        'vulnerable_cases':a['vulnerable_cases'],
                        'vulnerable_cases_pct':rate(a['vulnerable_cases'],a['case_count'])})
    boundary_shift_rows=[]
    for (pop,k,label),counts in sorted(evidence_gate_stats.items()):
        boundary_shift_rows.append({'population':pop,'error_count':k,'evidence_boundary_type':label,
                                    'feasible_scenarios':counts['scenarios'],
                                    'admission_changes':counts['flips'],
                                    'flip_pct':rate(counts['flips'],counts['scenarios'])})
    factor_rows=[]
    for factor, counts in factor_stats.items():
        factor_rows.append({'factor':factor,'dimension':DIM_BY_FACTOR[factor],
                            'HPS_per_1_level':round(WEIGHTS[DIM_BY_FACTOR[factor]]/len(FACTORS[DIM_BY_FACTOR[factor]]),8),
                            'feasible_single_errors':counts['scenarios'],
                            'admission_changes':counts['flips'],
                            'flip_pct':rate(counts['flips'],counts['scenarios'])})
    factor_rows.sort(key=lambda r:(-r['flip_pct'],r['factor']))
    baseline_main=sum(c['baseline']['accepted'] for c in main_cases)
    baseline_stress=sum(c['baseline']['accepted'] for c in stress_cases)
    summary_json={
        'title':'TADP simulated reviewer uncertainty: exhaustive 1, 2 and 3 factor ±1 perturbations',
        'policy':POLICY_VERSION,
        'authentic_uploaded_documents':0,'actual_reviewers':0,'federated_training_runs':0,
        'note':'Simulated rubric-level errors, not observed human error probabilities; no real documentary verification.',
        'main_cases':len(main_cases),'main_admitted':baseline_main,
        'boundary_enriched_cases':len(stress_cases),'boundary_enriched_admitted':baseline_stress,
        'boundary_target_categories':list(dict.fromkeys(c['goal'] for c in stress_cases)),
        'documentary_seed':EVIDENCE_SEED,'boundary_seed':ERROR_SEED+111,
        'D2_source':source,'D2_fixed_in_all_perturbations':True,
        'simulated_profiles_reuse_D2_contexts':bool(archived),
        'all_feasible_combinations_enumerated':True,
        'scalar_vector_consistency_checks':scalar_validation_checks,
        'threshold_floating_point_tolerance':1e-10,
        'per_assessment_nominal_combinations':{'1':40,'2':760,'3':9120},
        'experiment_results':summary,
        'evidence_boundary_subgroups':boundary_shift_rows,
        'boundary_window':{'HPS':NEAR_HPS_WINDOW,'review':NEAR_REVIEW_WINDOW,'dimension':NEAR_DIM_WINDOW},
        'conclusion_limit':'Policy sensitivity under a specified artificial perturbation distribution; not empirical rates of reviewer errors, document authenticity, or real-provider performance.'
    }
    write_csv(RESULTS_DIR/'Synthetic_Documentary_Library_1400.csv',library)
    write_csv(RESULTS_DIR/'Baseline_All_Cases.csv',baseline_rows)
    write_csv(RESULTS_DIR/'Scenario_Results.csv',summary)
    write_csv(RESULTS_DIR/'Case_Level_Sensitivity.csv',per_case)
    write_csv(RESULTS_DIR/'Single_Factor_Impact.csv',factor_rows)
    write_csv(RESULTS_DIR/'Evidence_Boundary_Impact.csv',boundary_shift_rows)
    write_csv(RESULTS_DIR/'Illustrative_Decision_Flips.csv',sample_flips)
    (RESULTS_DIR/'Summary.json').write_text(json.dumps(summary_json,indent=2)+'\n',encoding='utf-8')

    def group(pop, subgroup='ALL', k=1):
        return next((r for r in summary if r['population']==pop and r['subgroup']==subgroup
                    and r['error_count']==k),None)
    try:
        import matplotlib.pyplot as plt
        x=np.arange(3)
        fig,ax=plt.subplots(figsize=(8.2,4.9))
        width=0.34
        ax.bar(x-width/2,[group('main_1000',k=k)['flip_pct_of_scenarios'] for k in (1,2,3)],width,label='Original 1,000 profiles')
        ax.bar(x+width/2,[group('targeted_boundary',k=k)['flip_pct_of_scenarios'] for k in (1,2,3)],width,label='Boundary-enriched synthetic profiles')
        ax.set_xticks(x,['One factor','Two factors','Three factors'])
        ax.set_ylabel('Admission flips / feasible error combinations (%)')
        ax.set_title('TADP sensitivity to simultaneous ±1 reviewer-level errors')
        ax.legend()
        ax.grid(axis='y',alpha=.15)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR/'Figure_1_2_3_Error_Comparison.png',dpi=180)
        plt.close(fig)

        cats=['hps_3_0','hps_3_5','review_3_25','dimension_2_5','zero_gate']
        fig,ax=plt.subplots(figsize=(9.5,4.9))
        yy=np.arange(len(cats));height=.24
        for j,k in enumerate((1,2,3)):
            ax.barh(yy+(j-1)*height,[group('targeted_boundary',c,k)['flip_pct_of_scenarios'] for c in cats],height,label=f'{k} factor(s)')
        ax.set_yticks(yy,cats)
        ax.set_xlabel('Admission flips / feasible combinations (%)')
        ax.set_title('Deliberately boundary-enriched synthetic stress cases')
        ax.legend()
        ax.grid(axis='x',alpha=.15)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR/'Figure_Boundary_Gates.png',dpi=180)
        plt.close(fig)

        fig,ax=plt.subplots(figsize=(9,7.2))
        bars=factor_rows[::-1]
        ax.barh([r['factor'] for r in bars],[r['flip_pct'] for r in bars])
        ax.set_xlabel('Admission flips / feasible single-factor ±1 changes (%)')
        ax.set_title('Per-factor effects in the original 1,000 synthetic cases')
        ax.grid(axis='x',alpha=.15)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR/'Figure_Single_Factor_Ranking.png',dpi=180)
        plt.close(fig)
    except ImportError:
        print('Matplotlib unavailable: numeric results still created.')

    main_rows=[group('main_1000',k=k) for k in (1,2,3)]
    stress_rows=[group('targeted_boundary',k=k) for k in (1,2,3)]
    lines=[
        '# TADP-HPS28 Human-Reviewer Uncertainty: Simulated 1, 2 and 3 Factor Errors',
        '',
        '## Study design',
        '- **Zero real evidence uploads or human reviewers.** This is a deterministic simulated sensitivity experiment.',
        '- Main sample: 1,000 synthetic documentary profiles, with the original 52/33/10/5 percent profile mixture.',
        f'- Separate targeted stress sample: {len(stress_cases)} synthetic profiles intentionally placed near HPS, review, dimension and zero-factor gates.',
        '- Each hypothetical mistake alters a different documentary factor by exactly +1 or -1 within levels 0–5.',
        '- One-, two- and three-factor perturbations are exhaustively enumerated; the eight TRAIN-derived D2 scores are held fixed.',
        f'- D2 source: **{source["source"]}**. The original archive check count is {source.get("original_admission_matches",0)}.',
        '- Each baseline and alternative is evaluated using the TADP-HPS28/1.0 policy; no FL retraining is performed.',
        '- These scenario frequencies do NOT measure the probability that a human reviewer makes any mistake.',
        '',
        '## Original 1,000-case synthetic profiles',
        f'Baseline admission: {baseline_main} / {len(main_cases)}.',
        '',
        '| Changed factors | Feasible combinations | Admission flips | Flip rate (%) | Cases vulnerable |',
        '|---:|---:|---:|---:|---:|'
    ]
    for k,r in enumerate(main_rows,1):
        lines.append(f'| {k} | {r["feasible_scenarios"]:,} | {r["admission_changes"]:,} | {r["flip_pct_of_scenarios"]:.4f} | {r["vulnerable_cases"]:,} / {r["cases"]:,} |')
    lines += ['', '## Targeted synthetic boundary stress',
              f'Baseline admission: {baseline_stress} / {len(stress_cases)}.', '',
              '| Changed factors | Feasible combinations | Admission flips | Flip rate (%) | Cases vulnerable |',
              '|---:|---:|---:|---:|---:|']
    for k,r in enumerate(stress_rows,1):
        lines.append(f'| {k} | {r["feasible_scenarios"]:,} | {r["admission_changes"]:,} | {r["flip_pct_of_scenarios"]:.4f} | {r["vulnerable_cases"]:,} / {r["cases"]:,} |')
    lines+=['', '## Interpretation safeguards',
            '- **Very high stability across the original sample may be driven partly by hard rejection of cases with fixed D2 zeros or many documentary zeros.** Check `Scenario_Results.csv` for those subgroups.',
            '- The boundary-enriched sample is deliberately selected, and its flip rates must not be averaged with or treated as representative of the main sample.',
            '- Even fully exhaustive ±1 simulations do not validate actual documentary evidence or human reviewer accuracy.',
            '- Percentages describe equally weighted feasible perturbation combinations; they are not human-mistake probabilities.',
            '- The 1,000 Adult D2 archive contexts represent repeated 50-client simulations, not 1,000 independent real providers.',
            '- No FL reruns: the original 590 training-method evaluations remain unchanged.',
            '', '## Output files',
            '- `Synthetic_Documentary_Library_1400.csv` (synthetic profiles, explicit label)',
            '- `Baseline_All_Cases.csv` (baseline HPS, decision and gate indicators)',
            '- `Scenario_Results.csv` (grouped exhaustive perturbation results)',
            '- `Case_Level_Sensitivity.csv` (case-level scenarios and flips)',
            '- `Single_Factor_Impact.csv` (main-sample single-factor ranking)',
            '- `Evidence_Boundary_Impact.csv` (0→1 and 1→0 evidence-status errors separately)',
            '- `Illustrative_Decision_Flips.csv` (first 250 flipped scenarios)',
            '- `Summary.json` (machine-readable summary)',
            '- Three PNG charts and this report.',
            '', '## How to run',
            'Upload only this `.py` script to Colab, optionally also upload `TADP_HPS28_Adult_Phased_Results_v5.zip`, and run:',
            '```python', '%run TADP_Reviewer_1_2_3_Boundary_AUTO.py', '```',
            'No prompts, API calls, manually prepared evidence, or reviewer inputs. In Colab, the results ZIP downloads automatically.',
            '', 'If the original ZIP is absent, the script automatically labels its replacement D2 values as synthetic.',
            ''
    ]
    (RESULTS_DIR/'REPORT.md').write_text('\n'.join(lines),encoding='utf-8')

    with zipfile.ZipFile(RESULTS_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(RESULTS_DIR.iterdir()):
            if p.is_file():z.write(p,arcname=f'{RESULTS_DIR.name}/{p.name}')

    print('\n'+'='*88)
    print('TADP-HPS28: AUTOMATIC REVIEWER +/-1 ERRORS — EXHAUSTIVE 1, 2 AND 3 FACTORS')
    print('='*88)
    print(f'Main assessments:         {len(main_cases):,} SYNTHETIC (baseline admitted {baseline_main:,})')
    print(f'Extra boundary cases:     {len(stress_cases):,} SYNTHETIC (baseline admitted {baseline_stress:,})')
    print(f'Original Adult archive:   {source["source"]}; matched {source.get("original_admission_matches",0):,}/1,000')
    print(f'Scalar/vector crosschecks: {scalar_validation_checks:,} alternative scenarios matched')
    print('No documents/humans:      0 actual evidence packages / 0 real reviewers')
    print('\nMain sample (original 1,000):')
    for k,r in enumerate(main_rows,1):
        print(f'  {k} factor(s): {r["admission_changes"]:,} / {r["feasible_scenarios"]:,} flips ({r["flip_pct_of_scenarios"]:.4f}%), vulnerable {r["vulnerable_cases"]}/{r["cases"]}')
    print('\nIntentionally boundary-enriched sample:')
    for k,r in enumerate(stress_rows,1):
        print(f'  {k} factor(s): {r["admission_changes"]:,} / {r["feasible_scenarios"]:,} flips ({r["flip_pct_of_scenarios"]:.4f}%), vulnerable {r["vulnerable_cases"]}/{r["cases"]}')
    print(f'\nD2 fixed zeros (main cases): {sum(boundary_properties(c)["D2_zero"] for c in main_cases):,}/{len(main_cases)}')
    print('Results ZIP:', RESULTS_ZIP.resolve())
    print('Study report:', (RESULTS_DIR/'REPORT.md').resolve())
    print('NOTE: No empirical human error rate, evidence validation or FL reruns.')
    print('='*88)
    if 'google.colab' in sys.modules:
        try:
            from google.colab import files as colab_files
            colab_files.download(str(RESULTS_ZIP))
        except Exception as exc:
            print(f'Automatic download unavailable: {exc}; the ZIP is saved at {RESULTS_ZIP.resolve()}')
    return summary_json


def self_check():
    """Check vectorized alternative scoring against the scalar reference policy."""
    rng=random.Random(123)
    for _ in range(12):
        docs={f:rng.randint(0,5) for f in DOC_NAMES}
        d2={f:rng.randint(0,5) for f in D2_FACTORS}
        case={'docs':docs,'d2':d2,'scores':{**docs,**d2},
              'baseline':evaluate({**docs,**d2})}
        for k,vals in all_combinations().items():
            out=evaluate_exhaustively(case,vals,k)
            if not out['n']:continue
            choices=np.random.default_rng(99+k).choice(out['n'],min(15,out['n']),replace=False)
            for j in choices:
                altered=case['scores'].copy()
                for fi,d in zip(out['idx'][j],out['deltas'][j]):
                    altered[DOC_NAMES[int(fi)]]+=int(d)
                actual=evaluate(altered)
                if actual['accepted']!=bool(out['new_accepted'][j]):
                    raise AssertionError(f'vector/scalar policy mismatch k={k} {altered}')


def main():
    self_check()
    return run_all()


if __name__=='__main__':
    main()



TADP-HPS28: AUTOMATIC REVIEWER +/-1 ERRORS — EXHAUSTIVE 1, 2 AND 3 FACTORS
Main assessments:         1,000 SYNTHETIC (baseline admitted 462)
Extra boundary cases:     400 SYNTHETIC (baseline admitted 213)
Original Adult archive:   archived_Adult_TRAIN_D2; matched 1,000/1,000
Scalar/vector crosschecks: 8,766 alternative scenarios matched
No documents/humans:      0 actual evidence packages / 0 real reviewers

Main sample (original 1,000):
  1 factor(s): 53 / 34,400 flips (0.1541%), vulnerable 5/1000
  2 factor(s): 1,500 / 574,013 flips (0.2613%), vulnerable 20/1000
  3 factor(s): 23,836 / 6,173,535 flips (0.3861%), vulnerable 111/1000

Intentionally boundary-enriched sample:
  1 factor(s): 1,440 / 15,909 flips (9.0515%), vulnerable 240/400
  2 factor(s): 38,316 / 300,542 flips (12.7490%), vulnerable 275/400
  3 factor(s): 554,004 / 3,585,756 flips (15.4501%), vulnerable 372/400

D2 fixed zeros (main cases): 457/1000
Results ZIP: /content/TADP_Reviewer_Multi_Error_Results.zip
Study repo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>